# EEE6503 Project #3: Long-tail Visual Recognition

## Proposed Method: TALC++

This notebook implements **TALC++: Tail-Aware Logit-Calibrated Decoupled Learning** for long-tail CIFAR-100.

TALC++ is organized as a three-stage method:

1. **Tail-aware representation learning** with CE strong augmentation, LDAM-DRW, Balanced Softmax, or logit-adjusted CE.
2. **BN-safe decoupled classifier adaptation** with class-balanced classifier retraining while frozen BatchNorm layers remain in eval mode.
3. **Validation-gated group-wise calibration** with identity/no-calibration included in every grid so calibration is only applied when validation improves.

Non-negotiable constraints used by the implementation:

- Use only the provided imbalanced CIFAR datasets created by `IMBALANCECIFAR100`.
- Report all four scenarios: `exp/0.1`, `exp/0.01`, `step/0.1`, `step/0.01`.
- Keep `BATCH_SIZE = 128`.
- Keep the total final training budget per model at or below 200 epochs.
- Build validation data only from the imbalanced training split.
- Use the test split only in the final frozen evaluation cell.
- Save final weights, summaries, params, MACs/FLOPs, and report-ready artifacts.


## Imports and Global Config

Step 1 defines the shared configuration and safeguards. Later steps will add models, losses, training, calibration, and reporting on top of this foundation.


In [ ]:
import os
import os.path as osp
import json
import csv
import math
import random
import time
import copy
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
import torchvision
import torchvision.transforms as transforms

warnings.filterwarnings("ignore", category=UserWarning)

DATASET = "CIFAR100"
NUM_CLASSES = 100
BATCH_SIZE = 128
DEFAULT_SEED = 0
SMOKE_TEST = False
RUN_FINAL_TEST = False
FINAL_SUBMISSION_TRACK = "split_calibrated"

SCENARIOS = [
    ("exp", 0.1),
    ("exp", 0.01),
    ("step", 0.1),
    ("step", 0.01),
]
SEARCH_SEEDS = [0]
CONFIRM_SEEDS = [0, 1, 2]
MODEL_CANDIDATES = ["resnet32"]
OPTIONAL_MODEL_CANDIDATES = []

STAGE1_EPOCHS = 160
STAGE2_EPOCHS = 35
BN_ALIGN_EPOCHS = 5
DRW_START_EPOCH = 120
WARMUP_EPOCHS = 5
RUN_FULL_GRID = True

if SMOKE_TEST:
    SCENARIOS = [("exp", 0.1)]
    SEARCH_SEEDS = [0]
    CONFIRM_SEEDS = [0]
    MODEL_CANDIDATES = ["resnet32"]
    OPTIONAL_MODEL_CANDIDATES = []
    STAGE1_EPOCHS = 2
    STAGE2_EPOCHS = 1
    BN_ALIGN_EPOCHS = 0
    DRW_START_EPOCH = 1
    WARMUP_EPOCHS = 0
    RUN_FULL_GRID = False

TOTAL_FINAL_EPOCHS = STAGE1_EPOCHS + STAGE2_EPOCHS + BN_ALIGN_EPOCHS
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 2e-4
VAL_RATIO = 0.1
MIN_TRAIN_PER_CLASS = 2
NUM_WORKERS = 2 if SMOKE_TEST else 4
PIN_MEMORY = torch.cuda.is_available()

ROOT_DIR = Path(".")
DATA_DIR = ROOT_DIR / "data"
LOG_ROOT = ROOT_DIR / "logs"
RESULTS_DIR = ROOT_DIR / "results"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert BATCH_SIZE == 128, "The assignment requires BATCH_SIZE == 128."
assert TOTAL_FINAL_EPOCHS <= 200, "Total final model budget must be <= 200 epochs."
if not SMOKE_TEST:
    assert DATASET == "CIFAR100", "Final runs must use CIFAR-100."
assert FINAL_SUBMISSION_TRACK in {"split_calibrated", "full_train_fixed"}

print(f"DATASET={DATASET}, BATCH_SIZE={BATCH_SIZE}, SMOKE_TEST={SMOKE_TEST}")
print(f"Scenarios: {SCENARIOS}")
print(f"Epoch budget: stage1={STAGE1_EPOCHS}, stage2={STAGE2_EPOCHS}, bn_align={BN_ALIGN_EPOCHS}, total={TOTAL_FINAL_EPOCHS}")
print(f"Device: {DEVICE}")


## Reproducibility Helpers

These utilities centralize seeding and environment logging. Validation splits and data order use explicit seeds.


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def describe_environment(seed=None):
    print("PyTorch:", torch.__version__)
    print("Torchvision:", torchvision.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    if seed is not None:
        print("Random seed:", seed)


set_seed(DEFAULT_SEED)
describe_environment(DEFAULT_SEED)


## Dataset Classes

The provided imbalanced CIFAR classes are preserved. Validation and test isolation are handled by helper functions in later cells.


In [ ]:
# ---------------------------------------------------------------------------
# Long-tailed CIFAR datasets (after Cao et al., NeurIPS 2019 / LDAM-DRW)
# ---------------------------------------------------------------------------
class IMBALANCECIFAR10(torchvision.datasets.CIFAR10):
    cls_num = 10

    def __init__(self, root, imb_type='exp', imb_factor=0.01, rand_number=0,
                 train=True, transform=None, target_transform=None,
                 download=False):
        super(IMBALANCECIFAR10, self).__init__(
            root, train, transform, target_transform, download)
        np.random.seed(rand_number)
        img_num_list = self.get_img_num_per_cls(
            self.cls_num, imb_type, imb_factor)
        self.gen_imbalanced_data(img_num_list)

    def get_img_num_per_cls(self, cls_num, imb_type, imb_factor):
        img_max = len(self.data) / cls_num
        img_num_per_cls = []
        if imb_type == 'exp':
            for cls_idx in range(cls_num):
                num = img_max * (imb_factor ** (cls_idx / (cls_num - 1.0)))
                img_num_per_cls.append(int(num))
        elif imb_type == 'step':
            for cls_idx in range(cls_num // 2):
                img_num_per_cls.append(int(img_max))
            for cls_idx in range(cls_num - cls_num // 2):
                img_num_per_cls.append(int(img_max * imb_factor))
        else:
            img_num_per_cls.extend([int(img_max)] * cls_num)
        return img_num_per_cls

    def gen_imbalanced_data(self, img_num_per_cls):
        new_data = []
        new_targets = []
        targets_np = np.array(self.targets, dtype=np.int64)
        classes = np.unique(targets_np)
        self.num_per_cls_dict = dict()
        for the_class, the_img_num in zip(classes, img_num_per_cls):
            self.num_per_cls_dict[the_class] = the_img_num
            idx = np.where(targets_np == the_class)[0]
            np.random.shuffle(idx)
            selec_idx = idx[:the_img_num]
            new_data.append(self.data[selec_idx, ...])
            new_targets.extend([the_class] * the_img_num)
        new_data = np.vstack(new_data)
        self.data = new_data
        self.targets = new_targets

    def get_cls_num_list(self):
        cls_num_list = []
        for i in range(self.cls_num):
            cls_num_list.append(self.num_per_cls_dict[i])
        return cls_num_list


class IMBALANCECIFAR100(IMBALANCECIFAR10):
    """`IMBALANCECIFAR10` with the CIFAR-100 metadata."""
    base_folder = 'cifar-100-python'
    url = "https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz"
    filename = "cifar-100-python.tar.gz"
    tgz_md5 = 'eb9058c3a382ffc7106e4002c42a8d85'
    train_list = [
        ['train', '16019d7e3df5f24257cddd939b257f8d'],
    ]
    test_list = [
        ['test', 'f0ef6b0ae62326f3e7ffdfab6717acfc'],
    ]
    meta = {
        'filename': 'meta',
        'key': 'fine_label_names',
        'md5': '7973b15100ade9c7d40fb424638fde48',
    }
    cls_num = 100


## Transforms

Training transforms use CIFAR augmentation with optional RandAugment and Cutout. Validation and test transforms are deterministic.


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)


class Cutout:
    def __init__(self, n_holes=1, length=16):
        self.n_holes = n_holes
        self.length = length

    def __call__(self, img):
        if not torch.is_tensor(img):
            raise TypeError("Cutout expects a tensor image after ToTensor/Normalize.")
        h, w = img.size(1), img.size(2)
        mask = torch.ones((h, w), dtype=img.dtype, device=img.device)
        for _ in range(self.n_holes):
            y = random.randrange(h)
            x = random.randrange(w)
            y1 = max(0, y - self.length // 2)
            y2 = min(h, y + self.length // 2)
            x1 = max(0, x - self.length // 2)
            x2 = min(w, x + self.length // 2)
            mask[y1:y2, x1:x2] = 0
        return img * mask.unsqueeze(0)


def build_transforms(use_randaugment=True, use_cutout=True, use_random_erasing=False):
    train_ops = [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
    ]
    if use_randaugment and hasattr(transforms, "RandAugment"):
        train_ops.append(transforms.RandAugment(num_ops=2, magnitude=9))
    train_ops.extend([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    if use_cutout:
        train_ops.append(Cutout(n_holes=1, length=16))
    if use_random_erasing and hasattr(transforms, "RandomErasing"):
        train_ops.append(transforms.RandomErasing(p=0.25))

    eval_ops = [
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]
    return transforms.Compose(train_ops), transforms.Compose(eval_ops)


def inverse_normalize(tensor):
    mean = torch.tensor(CIFAR_MEAN, dtype=tensor.dtype, device=tensor.device).view(3, 1, 1)
    std = torch.tensor(CIFAR_STD, dtype=tensor.dtype, device=tensor.device).view(3, 1, 1)
    return tensor * std + mean


transform_train, transform_test = build_transforms()


## Protected Train/Validation Split

Validation is created only from the imbalanced training split. The train subset uses random augmentation, while validation uses deterministic evaluation transforms.


In [ ]:
def get_dataset_targets(dataset):
    if isinstance(dataset, Subset):
        base_targets = get_dataset_targets(dataset.dataset)
        return [base_targets[i] for i in dataset.indices]
    if hasattr(dataset, "targets"):
        return list(dataset.targets)
    if hasattr(dataset, "labels"):
        return list(dataset.labels)
    raise AttributeError("Dataset does not expose targets/labels.")


def count_classes(targets, num_classes=NUM_CLASSES):
    return np.bincount(np.asarray(targets, dtype=np.int64), minlength=num_classes).astype(int)


def make_protected_stratified_split(targets, val_ratio=0.1, seed=0,
                                    min_train_per_class=2,
                                    prefer_one_val_for_rare=True):
    rng = np.random.default_rng(seed)
    targets = np.asarray(targets, dtype=np.int64)
    train_indices = []
    val_indices = []

    for cls in np.unique(targets):
        cls_indices = np.where(targets == cls)[0]
        rng.shuffle(cls_indices)
        n = len(cls_indices)
        if n <= min_train_per_class:
            val_count = 0
        else:
            proposed = int(round(n * val_ratio))
            if prefer_one_val_for_rare:
                proposed = max(1, proposed)
            val_count = min(proposed, n - min_train_per_class)
        val_indices.extend(cls_indices[:val_count].tolist())
        train_indices.extend(cls_indices[val_count:].tolist())

    train_indices = np.asarray(train_indices, dtype=np.int64)
    val_indices = np.asarray(val_indices, dtype=np.int64)
    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    assert len(set(train_indices.tolist()).intersection(set(val_indices.tolist()))) == 0
    return train_indices.tolist(), val_indices.tolist()


def get_class_names(dataset):
    base = dataset.dataset if isinstance(dataset, Subset) else dataset
    return list(getattr(base, "classes", [str(i) for i in range(NUM_CLASSES)]))


def build_datasets_for_scenario(imb_type, imb_factor, seed=0, val_ratio=VAL_RATIO,
                                download=True, use_randaugment=True,
                                use_cutout=True, use_random_erasing=False):
    assert DATASET == "CIFAR100", "This project foundation is configured for CIFAR-100 final runs."
    train_transform, eval_transform = build_transforms(
        use_randaugment=use_randaugment,
        use_cutout=use_cutout,
        use_random_erasing=use_random_erasing,
    )
    train_aug_dataset = IMBALANCECIFAR100(
        root=str(DATA_DIR), imb_type=imb_type, imb_factor=imb_factor,
        rand_number=seed, train=True, download=download, transform=train_transform)
    train_eval_dataset = IMBALANCECIFAR100(
        root=str(DATA_DIR), imb_type=imb_type, imb_factor=imb_factor,
        rand_number=seed, train=True, download=download, transform=eval_transform)
    assert list(train_aug_dataset.targets) == list(train_eval_dataset.targets), "Train/eval transform datasets must share targets."

    full_targets = get_dataset_targets(train_aug_dataset)
    train_indices, val_indices = make_protected_stratified_split(
        full_targets, val_ratio=val_ratio, seed=seed,
        min_train_per_class=MIN_TRAIN_PER_CLASS)

    train_subset = Subset(train_aug_dataset, train_indices)
    val_subset = Subset(train_eval_dataset, val_indices)
    train_targets = [full_targets[i] for i in train_indices]
    val_targets = [full_targets[i] for i in val_indices]
    train_cls_num_list = count_classes(train_targets, NUM_CLASSES).tolist()
    val_cls_num_list = count_classes(val_targets, NUM_CLASSES).tolist()

    test_dataset = torchvision.datasets.CIFAR100(
        root=str(DATA_DIR), train=False, download=download, transform=eval_transform)

    return {
        "train_dataset": train_subset,
        "val_dataset": val_subset,
        "test_dataset": test_dataset,
        "train_targets": train_targets,
        "val_targets": val_targets,
        "test_targets": get_dataset_targets(test_dataset),
        "train_indices": train_indices,
        "val_indices": val_indices,
        "train_cls_num_list": train_cls_num_list,
        "val_cls_num_list": val_cls_num_list,
        "full_train_cls_num_list": train_aug_dataset.get_cls_num_list(),
        "class_names": get_class_names(train_aug_dataset),
    }


## Data Loaders and Samplers

Stage 1 uses natural sampling by default. Stage 2 will use class-balanced sampling through `WeightedRandomSampler`.


In [ ]:
def make_natural_loader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, drop_last=False):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=PIN_MEMORY,
        drop_last=drop_last,
    )


def make_balanced_sampler(targets, num_classes=NUM_CLASSES):
    targets = np.asarray(targets, dtype=np.int64)
    class_count = count_classes(targets, num_classes)
    sample_weights = np.array([1.0 / max(class_count[y], 1) for y in targets], dtype=np.float64)
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )


def make_balanced_loader(dataset, targets=None, batch_size=BATCH_SIZE,
                         num_workers=NUM_WORKERS, drop_last=False):
    if targets is None:
        targets = get_dataset_targets(dataset)
    sampler = make_balanced_sampler(targets, NUM_CLASSES)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=PIN_MEMORY,
        drop_last=drop_last,
    )


## Metrics and Evaluation

Validation selection uses a weighted score balancing overall, macro, and tail accuracy. Group definitions are based on training-subset frequency, not class IDs.


In [ ]:
def get_frequency_groups(cls_num_list):
    counts = np.asarray(cls_num_list, dtype=np.int64)
    num_classes = len(counts)
    order = np.argsort(-counts)
    head_count = min(33, num_classes)
    medium_count = min(34, max(0, num_classes - head_count))
    head = order[:head_count]
    medium = order[head_count:head_count + medium_count]
    tail = order[head_count + medium_count:]
    group_per_class = np.array(["unknown"] * num_classes, dtype=object)
    group_per_class[head] = "head"
    group_per_class[medium] = "medium"
    group_per_class[tail] = "tail"
    return {
        "head": head,
        "medium": medium,
        "tail": tail,
        "order": order,
        "group_per_class": group_per_class,
    }


def selection_score(metrics):
    return 0.55 * metrics["overall_acc"] + 0.25 * metrics["macro_acc"] + 0.20 * metrics["tail_acc"]


def _mean_per_class_acc(per_class_acc, per_class_count, class_indices):
    class_indices = np.asarray(class_indices, dtype=np.int64)
    if class_indices.size == 0:
        return 0.0
    present = class_indices[per_class_count[class_indices] > 0]
    if present.size == 0:
        return 0.0
    return float(np.nanmean(per_class_acc[present]))


def _sample_acc_for_classes(confusion, class_indices):
    class_indices = np.asarray(class_indices, dtype=np.int64)
    if class_indices.size == 0:
        return 0.0
    total = confusion[class_indices, :].sum()
    if total <= 0:
        return 0.0
    correct = np.trace(confusion[np.ix_(class_indices, class_indices)])
    return float(100.0 * correct / total)


def expected_calibration_error(confidences, correct, n_bins=15):
    if len(confidences) == 0:
        return 0.0
    confidences = np.asarray(confidences, dtype=np.float64)
    correct = np.asarray(correct, dtype=np.float64)
    ece = 0.0
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if not np.any(mask):
            continue
        ece += np.mean(mask) * abs(np.mean(correct[mask]) - np.mean(confidences[mask]))
    return float(100.0 * ece)


def apply_logit_calibration(logits, calibration=None, cls_num_list=None, groups=None):
    if calibration is None:
        return logits
    if cls_num_list is None:
        raise ValueError("cls_num_list is required for logit calibration.")
    counts = torch.tensor(cls_num_list, dtype=logits.dtype, device=logits.device)
    prior = counts / counts.sum().clamp_min(1.0)
    calibrated = logits.clone()

    if groups is None:
        groups = get_frequency_groups(cls_num_list)
    group_per_class = groups["group_per_class"]
    temperatures = torch.ones(logits.size(1), dtype=logits.dtype, device=logits.device)
    for group_name, key in [("head", "T_head"), ("medium", "T_medium"), ("tail", "T_tail")]:
        idx = np.where(group_per_class == group_name)[0]
        if len(idx) > 0:
            temperatures[torch.as_tensor(idx, dtype=torch.long, device=logits.device)] = float(calibration.get(key, 1.0))
    calibrated = calibrated / temperatures.clamp_min(1e-6)
    tau_la = float(calibration.get("logit_adjust_tau", 0.0))
    if tau_la != 0.0:
        calibrated = calibrated - tau_la * torch.log(prior.clamp_min(1e-12)).view(1, -1)
    return calibrated


def evaluate(model, loader, device, num_classes, cls_num_list=None,
             calibration=None, criterion=None, return_details=False):
    was_training = model.training
    model.eval()
    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)
    total_loss = 0.0
    total_seen = 0
    confidences = []
    correct_flags = []
    groups = get_frequency_groups(cls_num_list) if cls_num_list is not None else None

    with torch.no_grad():
        for image, target in loader:
            image = image.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            output = model(image)
            logits = output[0] if isinstance(output, tuple) else output
            logits = apply_logit_calibration(logits, calibration, cls_num_list, groups) if calibration else logits

            if criterion is not None:
                loss = criterion(logits, target)
                total_loss += float(loss.detach().cpu()) * target.numel()

            prob = torch.softmax(logits, dim=1)
            confidence, pred = prob.max(dim=1)
            confidences.extend(confidence.detach().cpu().tolist())
            correct_flags.extend(pred.eq(target).detach().cpu().tolist())

            t_np = target.detach().cpu().numpy()
            p_np = pred.detach().cpu().numpy()
            np.add.at(confusion, (t_np, p_np), 1)
            total_seen += target.numel()

    per_class_count = confusion.sum(axis=1)
    per_class_correct = np.diag(confusion)
    per_class_acc = np.full(num_classes, np.nan, dtype=np.float64)
    present = per_class_count > 0
    per_class_acc[present] = 100.0 * per_class_correct[present] / per_class_count[present]

    overall_acc = float(100.0 * per_class_correct.sum() / max(total_seen, 1))
    macro_acc = float(np.nanmean(per_class_acc[present])) if np.any(present) else 0.0

    if groups is None:
        split = num_classes // 3
        groups = {
            "head": np.arange(0, split),
            "medium": np.arange(split, 2 * split),
            "tail": np.arange(2 * split, num_classes),
            "group_per_class": np.array(["unknown"] * num_classes, dtype=object),
        }

    metrics = {
        "overall_acc": overall_acc,
        "macro_acc": macro_acc,
        "head_acc": _mean_per_class_acc(per_class_acc, per_class_count, groups["head"]),
        "medium_acc": _mean_per_class_acc(per_class_acc, per_class_count, groups["medium"]),
        "tail_acc": _mean_per_class_acc(per_class_acc, per_class_count, groups["tail"]),
        "head50_acc": _sample_acc_for_classes(confusion, np.arange(0, min(50, num_classes))),
        "tail50_acc": _sample_acc_for_classes(confusion, np.arange(min(50, num_classes), num_classes)),
        "per_class_acc": per_class_acc,
        "per_class_count": per_class_count,
        "confusion": confusion,
        "ece": expected_calibration_error(confidences, correct_flags),
    }
    metrics["selection_score"] = selection_score(metrics)

    if criterion is not None:
        metrics["avg_loss"] = float(total_loss / max(total_seen, 1))

    metrics["val_class_coverage"] = float(np.sum(present) / num_classes)
    for group_name in ["head", "medium", "tail"]:
        idx = groups[group_name]
        metrics[f"{group_name}_val_coverage"] = float(np.sum(per_class_count[idx] > 0) / max(len(idx), 1))

    if was_training:
        model.train()
    if return_details:
        metrics["groups"] = groups
    return metrics


## Parameters, MACs, and FLOPs

The project should not require external profiling packages. These hooks count convolution and linear MACs for CIFAR input size `(1, 3, 32, 32)`.


In [ ]:
def count_parameters_total(model):
    return sum(p.numel() for p in model.parameters())


def count_parameters_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def profile_macs_flops(model, input_size=(1, 3, 32, 32), device=None):
    if device is None:
        try:
            device = next(model.parameters()).device
        except StopIteration:
            device = DEVICE
    hooks = []
    totals = {"macs": 0}

    def conv_hook(module, inputs, output):
        x = inputs[0]
        batch = x.shape[0]
        out_h, out_w = output.shape[2], output.shape[3]
        kernel_h, kernel_w = module.kernel_size
        in_channels = module.in_channels
        out_channels = module.out_channels
        groups = module.groups
        totals["macs"] += int(batch * out_channels * out_h * out_w * (in_channels // groups) * kernel_h * kernel_w)

    def linear_hook(module, inputs, output):
        x = inputs[0]
        batch = x.shape[0] if x.dim() > 1 else 1
        totals["macs"] += int(batch * module.in_features * module.out_features)

    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            hooks.append(module.register_forward_hook(conv_hook))
        elif isinstance(module, nn.Linear):
            hooks.append(module.register_forward_hook(linear_hook))

    was_training = model.training
    model.eval()
    dummy = torch.zeros(input_size, device=device)
    with torch.no_grad():
        _ = model(dummy)
    for handle in hooks:
        handle.remove()
    if was_training:
        model.train()
    macs = totals["macs"]
    return {"macs": macs, "flops": 2 * macs}


def model_size_summary(model, input_size=(1, 3, 32, 32), device=None):
    profile = profile_macs_flops(model, input_size=input_size, device=device)
    params_total = count_parameters_total(model)
    params_trainable = count_parameters_trainable(model)
    return {
        "params_total": params_total,
        "params_trainable": params_trainable,
        "params_M": params_total / 1e6,
        "trainable_params_M": params_trainable / 1e6,
        "MACs": profile["macs"],
        "FLOPs": profile["flops"],
        "MACs_M": profile["macs"] / 1e6,
        "FLOPs_M": profile["flops"] / 1e6,
    }


## Artifact and Logging Helpers

Later steps will use these helpers to save candidate summaries, selected configs, checkpoints, and final reports.


In [ ]:
def ensure_base_dirs():
    LOG_ROOT.mkdir(parents=True, exist_ok=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def scenario_tag(imb_type, imb_factor):
    return f"scenario_{imb_type}_{str(imb_factor).replace('.', 'p')}"


def scenario_dir(imb_type, imb_factor):
    path = LOG_ROOT / scenario_tag(imb_type, imb_factor)
    path.mkdir(parents=True, exist_ok=True)
    return path


def candidate_dir(imb_type, imb_factor, candidate_id):
    path = scenario_dir(imb_type, imb_factor) / "candidates" / str(candidate_id)
    path.mkdir(parents=True, exist_ok=True)
    return path


def selected_dir(imb_type, imb_factor):
    path = scenario_dir(imb_type, imb_factor) / "selected"
    path.mkdir(parents=True, exist_ok=True)
    return path


def to_serializable(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if torch.is_tensor(obj):
        return obj.detach().cpu().tolist()
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_serializable(v) for v in obj]
    return obj


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        json.dump(to_serializable(payload), f, indent=2, sort_keys=True)


def read_json(path):
    with Path(path).open("r") as f:
        return json.load(f)


def write_csv(path, rows, fieldnames=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = [to_serializable(row) for row in rows]
    if fieldnames is None:
        keys = []
        for row in rows:
            for key in row.keys():
                if key not in keys:
                    keys.append(key)
        fieldnames = keys
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, "") for key in fieldnames})


def append_csv(path, row, fieldnames=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    row = to_serializable(row)
    if fieldnames is None:
        fieldnames = list(row.keys())
    write_header = not path.exists()
    with path.open("a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow({key: row.get(key, "") for key in fieldnames})


ensure_base_dirs()


## Model Definitions

Step 2 adds CIFAR-native ResNet backbones with the shared TALC++ interface. The main model is `CifarResNet32`; `CifarResNet56` is available for later ablations.


In [ ]:
class CifarResidualBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride,
                               padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out, inplace=True)


class CifarResNetBase(nn.Module):
    def __init__(self, num_blocks, num_classes=NUM_CLASSES):
        super().__init__()
        self.num_classes = num_classes
        self.feature_dim = 64
        self.in_planes = 16
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
        )
        self.layer1 = self._make_stage(16, num_blocks[0], stride=1)
        self.layer2 = self._make_stage(32, num_blocks[1], stride=2)
        self.layer3 = self._make_stage(64, num_blocks[2], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(self.feature_dim, num_classes)
        self._init_weights()

    def _make_stage(self, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for block_stride in strides:
            layers.append(CifarResidualBlock(self.in_planes, planes, block_stride))
            self.in_planes = planes * CifarResidualBlock.expansion
        return nn.Sequential(*layers)

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, 0, 0.01)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward_features(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)

    def forward_classifier(self, features):
        return self.classifier(features)

    def forward(self, x, return_features=False):
        features = self.forward_features(x)
        logits = self.forward_classifier(features)
        if return_features:
            return logits, features
        return logits

    def reset_classifier(self, num_classes=None):
        if num_classes is None:
            num_classes = self.num_classes
        old_classifier = self.classifier
        device = old_classifier.weight.device
        dtype = old_classifier.weight.dtype
        self.num_classes = num_classes
        self.classifier = nn.Linear(self.feature_dim, num_classes).to(device=device, dtype=dtype)
        nn.init.normal_(self.classifier.weight, 0, 0.01)
        if self.classifier.bias is not None:
            nn.init.zeros_(self.classifier.bias)

    def freeze_backbone(self, freeze_bn=True):
        for param in self.parameters():
            param.requires_grad = False
        for param in self.classifier.parameters():
            param.requires_grad = True
        if freeze_bn:
            self.freeze_batchnorm()

    def freeze_batchnorm(self):
        for module in self.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()

    def unfreeze_all(self):
        for param in self.parameters():
            param.requires_grad = True
        self.train()

    def get_classifier(self):
        return self.classifier

    def set_classifier(self, new_classifier):
        self.classifier = new_classifier
        if hasattr(new_classifier, "out_features"):
            self.num_classes = int(new_classifier.out_features)


class CifarResNet32(CifarResNetBase):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__([5, 5, 5], num_classes=num_classes)


class CifarResNet56(CifarResNetBase):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__([9, 9, 9], num_classes=num_classes)


MODEL_REGISTRY = {
    "resnet32": CifarResNet32,
    "cifar_resnet32": CifarResNet32,
    "CifarResNet32": CifarResNet32,
    "resnet56": CifarResNet56,
    "cifar_resnet56": CifarResNet56,
    "CifarResNet56": CifarResNet56,
}


def build_model(model_name, num_classes=NUM_CLASSES):
    if model_name not in MODEL_REGISTRY:
        available = ", ".join(sorted(MODEL_REGISTRY))
        raise ValueError(f"Unknown model_name={model_name!r}. Available: {available}")
    return MODEL_REGISTRY[model_name](num_classes=num_classes)


## Loss Functions and Classifier Helpers

These are the Step 2 building blocks for TALC++ Stage 1, Stage 2, and later calibration search. Training loops are intentionally deferred to Step 3.


In [ ]:
def build_ce_loss(weight=None):
    return nn.CrossEntropyLoss(weight=weight)


class LDAMLoss(nn.Module):
    def __init__(self, cls_num_list, max_m=0.5, s=30.0, weight=None):
        super().__init__()
        cls_counts = np.asarray(cls_num_list, dtype=np.float64)
        safe_counts = np.maximum(cls_counts, 1.0)
        margins = 1.0 / np.power(safe_counts, 0.25)
        margins = margins * (max_m / np.max(margins))
        self.register_buffer("m_list", torch.tensor(margins, dtype=torch.float32))
        self.s = float(s)
        self.weight = None
        self.set_weight(weight)

    def set_weight(self, weight):
        if weight is None:
            self.weight = None
        elif torch.is_tensor(weight):
            self.weight = weight.detach().clone().float()
        else:
            self.weight = torch.tensor(weight, dtype=torch.float32)

    def forward(self, logits, target):
        margins = self.m_list.to(logits.device)[target].view(-1, 1)
        target_logits = logits.gather(1, target.view(-1, 1))
        adjusted_logits = logits.clone()
        adjusted_logits.scatter_(1, target.view(-1, 1), target_logits - margins)
        weight = self.weight.to(logits.device) if self.weight is not None else None
        return F.cross_entropy(self.s * adjusted_logits, target, weight=weight)


def class_balanced_weights(cls_num_list, beta=0.9999):
    cls_num_array = np.asarray(cls_num_list, dtype=np.float64)
    effective_num = 1.0 - np.power(beta, cls_num_array)
    weights = (1.0 - beta) / np.maximum(effective_num, 1e-12)
    weights = weights / np.sum(weights) * len(cls_num_array)
    return torch.tensor(weights, dtype=torch.float32)


class BalancedSoftmaxLoss(nn.Module):
    def __init__(self, cls_num_list):
        super().__init__()
        log_counts = torch.log(torch.tensor(cls_num_list, dtype=torch.float32) + 1e-12)
        self.register_buffer("log_counts", log_counts)

    def forward(self, logits, target):
        return F.cross_entropy(logits + self.log_counts.to(logits.device), target)


def balanced_softmax_soft_loss(logits, soft_targets, cls_num_list):
    log_counts = torch.log(torch.tensor(cls_num_list, dtype=logits.dtype, device=logits.device) + 1e-12)
    log_prob = F.log_softmax(logits + log_counts.view(1, -1), dim=1)
    return -(soft_targets.to(logits.device) * log_prob).sum(dim=1).mean()


def class_prior_tensor(cls_num_list, device=None, dtype=torch.float32):
    counts = torch.tensor(cls_num_list, dtype=dtype, device=device)
    return counts / counts.sum().clamp_min(1.0)


def apply_train_logit_adjustment(logits, cls_num_list, tau_train=1.0):
    prior = class_prior_tensor(cls_num_list, device=logits.device, dtype=logits.dtype)
    return logits + float(tau_train) * torch.log(prior.clamp_min(1e-12)).view(1, -1)


def logit_adjusted_cross_entropy(logits, target, cls_num_list, tau_train=1.0):
    return F.cross_entropy(apply_train_logit_adjustment(logits, cls_num_list, tau_train), target)


class LabelAwareSmoothingCE(nn.Module):
    def __init__(self, cls_num_list, eps_min=0.0, eps_max=0.2):
        super().__init__()
        counts = np.asarray(cls_num_list, dtype=np.float64)
        n_min = float(np.min(counts))
        n_max = float(np.max(counts))
        normalized = (counts - n_min) / (n_max - n_min + 1e-12)
        eps = eps_min + (eps_max - eps_min) * normalized
        self.register_buffer("epsilons", torch.tensor(eps, dtype=torch.float32))

    def forward(self, logits, target):
        log_prob = F.log_softmax(logits, dim=1)
        eps = self.epsilons.to(logits.device)[target]
        nll = -log_prob.gather(1, target.view(-1, 1)).squeeze(1)
        smooth_loss = -log_prob.mean(dim=1)
        return ((1.0 - eps) * nll + eps * smooth_loss).mean()


def one_hot_targets(target, num_classes, dtype=torch.float32):
    return F.one_hot(target, num_classes=num_classes).to(dtype=dtype, device=target.device)


def mixup_data(x, y, alpha):
    if alpha is None or alpha <= 0.0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1.0 - lam) * x[index]
    return mixed_x, y, y[index], float(lam)


def rand_bbox(size, lam):
    _, _, height, width = size
    cut_ratio = math.sqrt(max(0.0, 1.0 - lam))
    cut_h = int(height * cut_ratio)
    cut_w = int(width * cut_ratio)
    cy = torch.randint(0, height, (1,)).item()
    cx = torch.randint(0, width, (1,)).item()
    y1 = max(cy - cut_h // 2, 0)
    y2 = min(cy + cut_h // 2, height)
    x1 = max(cx - cut_w // 2, 0)
    x2 = min(cx + cut_w // 2, width)
    return y1, x1, y2, x2


def cutmix_data(x, y, alpha):
    if alpha is None or alpha <= 0.0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0), device=x.device)
    y1, x1, y2, x2 = rand_bbox(x.size(), lam)
    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    area = max(y2 - y1, 0) * max(x2 - x1, 0)
    lam = 1.0 - area / float(x.size(2) * x.size(3))
    return mixed_x, y, y[index], float(lam)


def mixed_soft_targets(y_a, y_b, lam, num_classes):
    return lam * one_hot_targets(y_a, num_classes) + (1.0 - lam) * one_hot_targets(y_b, num_classes)


def clone_classifier_state(classifier):
    return {key: value.detach().clone() for key, value in classifier.state_dict().items()}


def restore_classifier_state(classifier, state):
    classifier.load_state_dict(copy.deepcopy(state))


def classifier_weight_norms(classifier):
    if not hasattr(classifier, "weight"):
        raise TypeError("Expected classifier with a weight tensor.")
    return classifier.weight.detach().norm(p=2, dim=1)


def apply_maxnorm_to_classifier(classifier, radius):
    if radius is None:
        return
    if radius <= 0:
        raise ValueError("MaxNorm radius must be positive or None.")
    if not hasattr(classifier, "weight"):
        raise TypeError("Expected classifier with a weight tensor.")
    with torch.no_grad():
        weight = classifier.weight.data
        norms = weight.norm(p=2, dim=1, keepdim=True).clamp_min(1e-12)
        desired = torch.clamp(norms, max=float(radius))
        weight.mul_(desired / norms)


def set_backbone_eval_classifier_train(model):
    if hasattr(model, "freeze_backbone"):
        model.freeze_backbone(freeze_bn=True)
    else:
        for param in model.parameters():
            param.requires_grad = False
    model.eval()
    classifier = model.get_classifier() if hasattr(model, "get_classifier") else model.classifier
    for param in classifier.parameters():
        param.requires_grad = True
    classifier.train()
    return classifier


def make_group_index_tensors(cls_num_list, device=None):
    groups = get_frequency_groups(cls_num_list)
    return {
        "head": torch.as_tensor(groups["head"], dtype=torch.long, device=device),
        "medium": torch.as_tensor(groups["medium"], dtype=torch.long, device=device),
        "tail": torch.as_tensor(groups["tail"], dtype=torch.long, device=device),
        "order": torch.as_tensor(groups["order"], dtype=torch.long, device=device),
    }


def default_calibration():
    return {
        "tau_norm": 0.0,
        "logit_adjust_tau": 0.0,
        "T_head": 1.0,
        "T_medium": 1.0,
        "T_tail": 1.0,
    }


def group_temperature_vector(num_classes, groups, calibration, device=None, dtype=torch.float32):
    temps = torch.ones(num_classes, dtype=dtype, device=device)
    group_per_class = groups["group_per_class"]
    for group_name, key in [("head", "T_head"), ("medium", "T_medium"), ("tail", "T_tail")]:
        indices = np.where(group_per_class == group_name)[0]
        if len(indices) > 0:
            temps[torch.as_tensor(indices, dtype=torch.long, device=device)] = float(calibration.get(key, 1.0))
    return temps


def calibrate_logits_groupwise(logits, cls_num_list, calibration=None, groups=None, eps=1e-12):
    if calibration is None:
        calibration = default_calibration()
    if groups is None:
        groups = get_frequency_groups(cls_num_list)
    temps = group_temperature_vector(
        logits.size(1), groups, calibration, device=logits.device, dtype=logits.dtype)
    calibrated = logits / temps.clamp_min(1e-6).view(1, -1)
    tau_la = float(calibration.get("logit_adjust_tau", 0.0))
    if tau_la != 0.0:
        prior = class_prior_tensor(cls_num_list, device=logits.device, dtype=logits.dtype)
        calibrated = calibrated - tau_la * torch.log(prior.clamp_min(eps)).view(1, -1)
    return calibrated


def apply_tau_normalization_to_classifier(classifier, tau_norm, eps=1e-12):
    original_state = clone_classifier_state(classifier)
    tau_norm = float(tau_norm)
    if tau_norm == 0.0:
        return original_state
    if not hasattr(classifier, "weight"):
        raise TypeError("Expected classifier with a weight tensor.")
    with torch.no_grad():
        weight = classifier.weight.data
        norms = weight.norm(p=2, dim=1, keepdim=True).clamp_min(eps)
        weight.copy_(weight / norms.pow(tau_norm))
    return original_state


def calibration_strength(calibration):
    identity = default_calibration()
    return sum(abs(float(calibration.get(key, value)) - float(value)) for key, value in identity.items())


## Step 2 Sanity Check Helper

This helper is intentionally not run automatically. It can be called after the notebook environment has `numpy`, `torch`, and `torchvision` available.


In [ ]:
def run_step2_sanity_checks(device=DEVICE):
    model = build_model("resnet32", NUM_CLASSES).to(device)
    x = torch.randn(2, 3, 32, 32, device=device)
    logits = model(x)
    assert logits.shape == (2, NUM_CLASSES), logits.shape
    logits_with_features, features = model(x, return_features=True)
    assert logits_with_features.shape == (2, NUM_CLASSES)
    assert features.ndim == 2 and features.shape[0] == 2
    size_info = model_size_summary(model, device=device)

    fake_cls_num_list = [10 + (i % 5) for i in range(NUM_CLASSES)]
    fake_logits = torch.randn(4, NUM_CLASSES, device=device)
    fake_targets = torch.tensor([0, 1, 2, 3], dtype=torch.long, device=device)

    ce_loss = build_ce_loss()(fake_logits, fake_targets)
    ldam_loss = LDAMLoss(fake_cls_num_list)(fake_logits, fake_targets)
    bs_loss = BalancedSoftmaxLoss(fake_cls_num_list)(fake_logits, fake_targets)
    las_loss = LabelAwareSmoothingCE(fake_cls_num_list)(fake_logits, fake_targets)
    la_loss = logit_adjusted_cross_entropy(fake_logits, fake_targets, fake_cls_num_list, tau_train=1.0)

    y_soft = one_hot_targets(fake_targets, NUM_CLASSES)
    bs_soft_loss = balanced_softmax_soft_loss(fake_logits, y_soft, fake_cls_num_list)

    classifier = model.get_classifier()
    original_state = apply_tau_normalization_to_classifier(classifier, tau_norm=0.5)
    restore_classifier_state(classifier, original_state)
    apply_maxnorm_to_classifier(classifier, radius=2.0)
    set_backbone_eval_classifier_train(model)

    return {
        "logits_shape": tuple(logits.shape),
        "features_shape": tuple(features.shape),
        "params_M": size_info["params_M"],
        "MACs_M": size_info["MACs_M"],
        "losses": {
            "ce": float(ce_loss.detach().cpu()),
            "ldam": float(ldam_loss.detach().cpu()),
            "balanced_softmax": float(bs_loss.detach().cpu()),
            "label_aware_smoothing": float(las_loss.detach().cpu()),
            "logit_adjusted": float(la_loss.detach().cpu()),
            "balanced_softmax_soft": float(bs_soft_loss.detach().cpu()),
        },
    }


## Stage 1 and Stage 2 Training

Step 3 implements the core TALC++ training pipeline: tail-aware representation learning followed by BN-safe classifier adaptation. Full scenario orchestration and calibration search are still deferred to Step 4.


In [ ]:
def assert_training_budget(config):
    stage1_epochs = int(config.get("stage1_epochs", STAGE1_EPOCHS))
    stage2_epochs = int(config.get("stage2_epochs", STAGE2_EPOCHS))
    bn_align_epochs = int(config.get("bn_align_epochs", BN_ALIGN_EPOCHS))
    total_epochs = stage1_epochs + stage2_epochs + bn_align_epochs
    assert total_epochs <= 200, f"Training budget exceeds 200 epochs: {total_epochs}"
    return total_epochs


def resolve_training_run_dir(config):
    if config.get("run_dir") is not None:
        path = Path(config["run_dir"])
    elif config.get("output_dir") is not None:
        path = Path(config["output_dir"])
    elif all(key in config for key in ["imb_type", "imb_factor", "candidate_id"]):
        path = candidate_dir(config["imb_type"], config["imb_factor"], config["candidate_id"])
    else:
        path = RESULTS_DIR / "standalone_training"
    path.mkdir(parents=True, exist_ok=True)
    return path


def scalar_metrics_for_log(metrics, prefix=""):
    scalar_keys = [
        "overall_acc", "macro_acc", "head_acc", "medium_acc", "tail_acc",
        "head50_acc", "tail50_acc", "ece", "avg_loss", "selection_score",
        "val_class_coverage", "head_val_coverage", "medium_val_coverage", "tail_val_coverage",
    ]
    out = {}
    for key in scalar_keys:
        if key in metrics:
            value = metrics[key]
            if isinstance(value, (int, float, np.integer, np.floating)):
                out[prefix + key] = float(value)
    return out


def metrics_without_large_arrays(metrics):
    skip = {"confusion", "groups"}
    out = {}
    for key, value in metrics.items():
        if key in skip:
            continue
        if key == "per_class_acc":
            out[key] = np.nan_to_num(value, nan=-1.0).tolist()
        elif key == "per_class_count":
            out[key] = np.asarray(value).astype(int).tolist()
        else:
            out[key] = to_serializable(value)
    return out


def state_dict_to_cpu(state_dict):
    cpu_state = {}
    for key, value in state_dict.items():
        if torch.is_tensor(value):
            cpu_state[key] = value.detach().cpu().clone()
        else:
            cpu_state[key] = copy.deepcopy(value)
    return cpu_state


def safe_model_size_summary(model, device):
    try:
        return model_size_summary(model, device=device)
    except Exception as exc:
        return {
            "params_total": count_parameters_total(model),
            "params_trainable": count_parameters_trainable(model),
            "params_M": count_parameters_total(model) / 1e6,
            "trainable_params_M": count_parameters_trainable(model) / 1e6,
            "MACs": None,
            "FLOPs": None,
            "MACs_M": None,
            "FLOPs_M": None,
            "profile_error": str(exc),
        }


def warmup_cosine_lr(base_lr, epoch, total_epochs, warmup_epochs=0, min_lr=0.0):
    if total_epochs <= 0:
        return base_lr
    if warmup_epochs > 0 and epoch < warmup_epochs:
        return base_lr * float(epoch + 1) / float(max(1, warmup_epochs))
    decay_epochs = max(1, total_epochs - warmup_epochs)
    progress = min(1.0, max(0.0, float(epoch - warmup_epochs) / float(decay_epochs)))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_lr + (base_lr - min_lr) * cosine


def set_optimizer_lr(optimizer, lr):
    for group in optimizer.param_groups:
        group["lr"] = lr


def build_sgd_optimizer(params, lr, weight_decay, momentum=MOMENTUM, nesterov=True):
    params = [p for p in params if p.requires_grad]
    if not params:
        raise ValueError("No trainable parameters were provided to the optimizer.")
    return torch.optim.SGD(
        params,
        lr=float(lr),
        momentum=float(momentum),
        weight_decay=float(weight_decay),
        nesterov=bool(nesterov),
    )


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.ema = copy.deepcopy(model).eval()
        self.decay = float(decay)
        for param in self.ema.parameters():
            param.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        model_state = model.state_dict()
        ema_state = self.ema.state_dict()
        for key, ema_value in ema_state.items():
            model_value = model_state[key].detach()
            if torch.is_floating_point(ema_value):
                ema_value.copy_(ema_value * self.decay + model_value * (1.0 - self.decay))
            else:
                ema_value.copy_(model_value)

    def copy_to(self, model):
        model.load_state_dict(self.ema.state_dict())

    def state_dict(self):
        return self.ema.state_dict()


def autocast_context(device):
    return torch.cuda.amp.autocast(enabled=(device.type == "cuda"))


def make_grad_scaler(device):
    return torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


def save_training_checkpoint(path, model, config, cls_num_list, epoch, val_metrics,
                             stage_name, extra=None, device=DEVICE):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "model_state_dict": state_dict_to_cpu(model.state_dict()),
        "model_name": config.get("model_name", config.get("model", "unknown")),
        "num_classes": len(cls_num_list),
        "stage_name": stage_name,
        "stage1_mode": config.get("stage1_mode"),
        "stage2_mode": config.get("stage2_mode"),
        "cls_num_list": list(map(int, cls_num_list)),
        "epoch": int(epoch),
        "val_metrics": metrics_without_large_arrays(val_metrics),
        "selection_score": float(val_metrics.get("selection_score", selection_score(val_metrics))),
        "config": to_serializable(config),
    }
    payload.update(safe_model_size_summary(model, device=device))
    if extra:
        for key, value in extra.items():
            payload[key] = value
    torch.save(payload, path)
    return str(path)


def load_training_checkpoint(model, checkpoint_path, device=DEVICE):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    return checkpoint


def write_stage_metrics(run_dir, name, metrics):
    write_json(Path(run_dir) / f"{name}.json", metrics_without_large_arrays(metrics))


def maybe_reset_log(log_path, overwrite=True):
    log_path = Path(log_path)
    if overwrite and log_path.exists():
        log_path.unlink()


def epoch_log_fieldnames(rows):
    keys = []
    for row in rows:
        for key in row.keys():
            if key not in keys:
                keys.append(key)
    return keys


### Stage 1: Tail-aware Representation Learning

Supported Stage 1 modes are `CE_StrongAug`, `LDAM_DRW`, `BalancedSoftmax`, and `LogitAdjustedCE`. BCL is intentionally deferred.


In [ ]:
def build_stage1_criterion(config, cls_num_list):
    stage1_mode = config.get("stage1_mode", "CE_StrongAug")
    if stage1_mode == "CE_StrongAug":
        return build_ce_loss()
    if stage1_mode == "LDAM_DRW":
        return LDAMLoss(
            cls_num_list,
            max_m=float(config.get("max_m", 0.5)),
            s=float(config.get("s", 30.0)),
            weight=None,
        )
    if stage1_mode == "BalancedSoftmax":
        return BalancedSoftmaxLoss(cls_num_list)
    if stage1_mode == "LogitAdjustedCE":
        return None
    raise ValueError(f"Unsupported Stage 1 mode in Step 3: {stage1_mode}")


def stage1_batch_loss(model, image, target, criterion, config, cls_num_list):
    stage1_mode = config.get("stage1_mode", "CE_StrongAug")
    if stage1_mode == "BalancedSoftmax":
        cutmix_alpha = float(config.get("cutmix_alpha", 0.0))
        mixup_alpha = float(config.get("mixup_alpha", 0.0))
        if cutmix_alpha > 0.0:
            mixed_image, y_a, y_b, lam = cutmix_data(image, target, cutmix_alpha)
            logits = model(mixed_image)
            soft_targets = mixed_soft_targets(y_a, y_b, lam, len(cls_num_list))
            return balanced_softmax_soft_loss(logits, soft_targets, cls_num_list)
        if mixup_alpha > 0.0:
            mixed_image, y_a, y_b, lam = mixup_data(image, target, mixup_alpha)
            logits = model(mixed_image)
            soft_targets = mixed_soft_targets(y_a, y_b, lam, len(cls_num_list))
            return balanced_softmax_soft_loss(logits, soft_targets, cls_num_list)

    logits = model(image)
    if stage1_mode == "LogitAdjustedCE":
        return logit_adjusted_cross_entropy(
            logits, target, cls_num_list, tau_train=float(config.get("tau_train", 1.0)))
    return criterion(logits, target)


def train_stage1_representation(config, train_loader, val_loader, model, cls_num_list, device):
    config = dict(config or {})
    config.setdefault("stage1_mode", "CE_StrongAug")
    config.setdefault("stage1_epochs", STAGE1_EPOCHS)
    config.setdefault("stage2_epochs", STAGE2_EPOCHS)
    config.setdefault("bn_align_epochs", BN_ALIGN_EPOCHS)
    assert_training_budget(config)

    run_dir = resolve_training_run_dir(config)
    log_path = run_dir / "train_log_stage1.csv"
    maybe_reset_log(log_path, overwrite=bool(config.get("overwrite_logs", True)))

    stage1_epochs = int(config.get("stage1_epochs", STAGE1_EPOCHS))
    warmup_epochs = int(config.get("warmup_epochs", WARMUP_EPOCHS))
    base_lr = float(config.get("lr", LR))
    min_lr = float(config.get("min_lr", 0.0))
    weight_decay = float(config.get("weight_decay", WEIGHT_DECAY))
    eval_interval = int(config.get("eval_interval", 1 if SMOKE_TEST else 5))
    use_ema = bool(config.get("use_ema", True))
    ema_decay = float(config.get("ema_decay", 0.999))
    stage1_mode = config["stage1_mode"]
    drw_start_epoch = int(config.get("drw_start_epoch", DRW_START_EPOCH))
    beta = float(config.get("beta", 0.9999))

    model.to(device)
    model.unfreeze_all() if hasattr(model, "unfreeze_all") else model.train()
    criterion = build_stage1_criterion(config, cls_num_list)
    cb_weights = class_balanced_weights(cls_num_list, beta=beta) if stage1_mode == "LDAM_DRW" else None
    optimizer = build_sgd_optimizer(model.parameters(), lr=base_lr, weight_decay=weight_decay)
    scaler = make_grad_scaler(device)
    ema = ModelEMA(model, decay=ema_decay) if use_ema else None

    log_rows = []
    best_score = -float("inf")
    best_path = run_dir / "best_stage1.pth"
    best_val_metrics = None
    best_epoch = -1
    best_source = "raw"

    for epoch in range(stage1_epochs):
        lr = warmup_cosine_lr(base_lr, epoch, stage1_epochs, warmup_epochs, min_lr=min_lr)
        set_optimizer_lr(optimizer, lr)
        model.train()
        if stage1_mode == "LDAM_DRW":
            if epoch >= drw_start_epoch:
                criterion.set_weight(cb_weights)
            else:
                criterion.set_weight(None)

        running_loss = 0.0
        seen = 0
        start_time = time.time()
        for image, target in train_loader:
            image = image.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast_context(device):
                loss = stage1_batch_loss(model, image, target, criterion, config, cls_num_list)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            if ema is not None:
                ema.update(model)
            batch_size = target.size(0)
            running_loss += float(loss.detach().cpu()) * batch_size
            seen += batch_size

        train_loss = running_loss / max(seen, 1)
        row = {
            "stage": "stage1",
            "epoch": epoch + 1,
            "lr": lr,
            "train_loss": train_loss,
            "elapsed_sec": time.time() - start_time,
            "stage1_mode": stage1_mode,
        }

        should_eval = ((epoch + 1) % eval_interval == 0) or (epoch == stage1_epochs - 1) or SMOKE_TEST
        if should_eval:
            val_metrics_raw = evaluate(
                model, val_loader, device, len(cls_num_list), cls_num_list=cls_num_list,
                criterion=criterion if criterion is not None else None)
            candidate_model = model
            candidate_metrics = val_metrics_raw
            candidate_source = "raw"
            row.update(scalar_metrics_for_log(val_metrics_raw, prefix="val_raw_"))

            if ema is not None:
                val_metrics_ema = evaluate(
                    ema.ema, val_loader, device, len(cls_num_list), cls_num_list=cls_num_list,
                    criterion=criterion if criterion is not None else None)
                row.update(scalar_metrics_for_log(val_metrics_ema, prefix="val_ema_"))
                if val_metrics_ema["selection_score"] > val_metrics_raw["selection_score"]:
                    candidate_model = ema.ema
                    candidate_metrics = val_metrics_ema
                    candidate_source = "ema"

            row.update(scalar_metrics_for_log(candidate_metrics, prefix="val_"))
            candidate_score = float(candidate_metrics["selection_score"])
            if candidate_score > best_score:
                best_score = candidate_score
                best_val_metrics = candidate_metrics
                best_epoch = epoch + 1
                best_source = candidate_source
                extra = {
                    "use_ema_stage1": candidate_source == "ema",
                    "best_source": candidate_source,
                    "ema_state_dict": state_dict_to_cpu(ema.state_dict()) if ema is not None else None,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "stage1_epochs": stage1_epochs,
                    "drw_start_epoch": drw_start_epoch,
                }
                save_training_checkpoint(
                    best_path, candidate_model, config, cls_num_list, epoch + 1,
                    candidate_metrics, stage_name="stage1", extra=extra, device=device)
                write_stage_metrics(run_dir, "val_metrics_stage1", candidate_metrics)

        log_rows.append(row)
        write_csv(log_path, log_rows, fieldnames=epoch_log_fieldnames(log_rows))
        print(
            f"[Stage 1][{stage1_mode}] epoch {epoch + 1}/{stage1_epochs} "
            f"loss={train_loss:.4f} best={best_score:.3f} source={best_source}")

    if best_val_metrics is None:
        best_val_metrics = evaluate(model, val_loader, device, len(cls_num_list), cls_num_list=cls_num_list)
        best_score = float(best_val_metrics["selection_score"])
        best_epoch = stage1_epochs
        save_training_checkpoint(
            best_path, model, config, cls_num_list, best_epoch, best_val_metrics,
            stage_name="stage1", extra={"use_ema_stage1": False, "best_source": "raw"}, device=device)
        write_stage_metrics(run_dir, "val_metrics_stage1", best_val_metrics)

    load_training_checkpoint(model, best_path, device=device)
    model.eval()
    return {
        "model": model,
        "run_dir": str(run_dir),
        "best_checkpoint_path": str(best_path),
        "train_log_path": str(log_path),
        "best_val_metrics": best_val_metrics,
        "best_selection_score": best_score,
        "best_epoch": best_epoch,
        "use_ema_stage1": best_source == "ema",
        "best_source": best_source,
    }


### Stage 2: BN-safe Decoupled Classifier Adaptation

Supported Stage 2 modes are `none`, `cRT_no_reset`, `cRT_reset`, `cRT_no_reset_LabelAwareSmoothing`, and `cRT_no_reset_MaxNorm`. Frozen BatchNorm layers remain in eval mode during classifier adaptation.


In [ ]:
def build_stage2_criterion(config, cls_num_list):
    stage2_mode = config.get("stage2_mode", "cRT_no_reset")
    loss_name = config.get("stage2_loss")
    if loss_name is None:
        if "LabelAwareSmoothing" in stage2_mode:
            loss_name = "LabelAwareSmoothingCE"
        elif config.get("loss") is not None:
            loss_name = config.get("loss")
        else:
            loss_name = "CE"

    if loss_name == "CE":
        return build_ce_loss(), loss_name
    if loss_name == "LabelAwareSmoothingCE":
        return LabelAwareSmoothingCE(
            cls_num_list,
            eps_min=float(config.get("eps_min", 0.0)),
            eps_max=float(config.get("eps_max", 0.2)),
        ), loss_name
    if loss_name == "BalancedSoftmax":
        return BalancedSoftmaxLoss(cls_num_list), loss_name
    raise ValueError(f"Unsupported Stage 2 loss: {loss_name}")


def stage2_batch_loss(model, image, target, criterion, config, cls_num_list, loss_name):
    mixup_alpha = float(config.get("stage2_mixup_alpha", 0.0))
    if loss_name == "BalancedSoftmax" and mixup_alpha > 0.0:
        mixed_image, y_a, y_b, lam = mixup_data(image, target, mixup_alpha)
        logits = model(mixed_image)
        soft_targets = mixed_soft_targets(y_a, y_b, lam, len(cls_num_list))
        return balanced_softmax_soft_loss(logits, soft_targets, cls_num_list)
    logits = model(image)
    return criterion(logits, target)


def save_stage2_none_checkpoint(config, model, val_loader, cls_num_list, device, run_dir):
    metrics = evaluate(model, val_loader, device, len(cls_num_list), cls_num_list=cls_num_list)
    best_path = run_dir / "best_stage2.pth"
    save_training_checkpoint(
        best_path, model, config, cls_num_list, 0, metrics, stage_name="stage2",
        extra={"stage2_mode": "none", "stage2_epochs": 0}, device=device)
    write_stage_metrics(run_dir, "val_metrics_stage2", metrics)
    return {
        "model": model,
        "run_dir": str(run_dir),
        "best_checkpoint_path": str(best_path),
        "train_log_path": None,
        "best_val_metrics": metrics,
        "best_selection_score": float(metrics["selection_score"]),
        "best_epoch": 0,
        "stage2_mode": "none",
    }


def train_stage2_classifier(config, model, train_balanced_loader, val_loader, cls_num_list, device):
    config = dict(config or {})
    config.setdefault("stage1_epochs", STAGE1_EPOCHS)
    config.setdefault("stage2_epochs", STAGE2_EPOCHS)
    config.setdefault("bn_align_epochs", BN_ALIGN_EPOCHS)
    config.setdefault("stage2_mode", "cRT_no_reset")
    assert_training_budget(config)

    run_dir = resolve_training_run_dir(config)
    stage2_mode = config["stage2_mode"]
    model.to(device)

    if stage2_mode == "none" or int(config.get("stage2_epochs", STAGE2_EPOCHS)) <= 0:
        return save_stage2_none_checkpoint(config, model, val_loader, cls_num_list, device, run_dir)

    if stage2_mode == "cRT_reset":
        model.reset_classifier(len(cls_num_list))
        model.to(device)

    classifier = set_backbone_eval_classifier_train(model)
    criterion, loss_name = build_stage2_criterion(config, cls_num_list)
    criterion = criterion.to(device) if isinstance(criterion, nn.Module) else criterion

    stage2_epochs = int(config.get("stage2_epochs", STAGE2_EPOCHS))
    base_lr = float(config.get("crt_lr", config.get("stage2_lr", 0.05)))
    weight_decay = float(config.get("crt_weight_decay", config.get("stage2_weight_decay", 0.0)))
    warmup_epochs = int(config.get("stage2_warmup_epochs", 0))
    min_lr = float(config.get("stage2_min_lr", 0.0))
    eval_interval = int(config.get("stage2_eval_interval", 1 if SMOKE_TEST else 5))
    maxnorm_radius = config.get("maxnorm_radius")
    if stage2_mode == "cRT_no_reset_MaxNorm" and maxnorm_radius is None:
        maxnorm_radius = 2.0

    optimizer = build_sgd_optimizer(classifier.parameters(), lr=base_lr, weight_decay=weight_decay)
    scaler = make_grad_scaler(device)
    log_path = run_dir / "train_log_stage2.csv"
    maybe_reset_log(log_path, overwrite=bool(config.get("overwrite_logs", True)))

    log_rows = []
    best_score = -float("inf")
    best_path = run_dir / "best_stage2.pth"
    best_val_metrics = None
    best_epoch = -1

    for epoch in range(stage2_epochs):
        lr = warmup_cosine_lr(base_lr, epoch, stage2_epochs, warmup_epochs, min_lr=min_lr)
        set_optimizer_lr(optimizer, lr)
        set_backbone_eval_classifier_train(model)

        running_loss = 0.0
        seen = 0
        start_time = time.time()
        for image, target in train_balanced_loader:
            image = image.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            set_backbone_eval_classifier_train(model)
            with autocast_context(device):
                loss = stage2_batch_loss(model, image, target, criterion, config, cls_num_list, loss_name)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            if maxnorm_radius is not None:
                apply_maxnorm_to_classifier(model.get_classifier(), float(maxnorm_radius))
            batch_size = target.size(0)
            running_loss += float(loss.detach().cpu()) * batch_size
            seen += batch_size

        train_loss = running_loss / max(seen, 1)
        row = {
            "stage": "stage2",
            "epoch": epoch + 1,
            "lr": lr,
            "train_loss": train_loss,
            "elapsed_sec": time.time() - start_time,
            "stage2_mode": stage2_mode,
            "stage2_loss": loss_name,
            "maxnorm_radius": maxnorm_radius if maxnorm_radius is not None else "",
        }

        should_eval = ((epoch + 1) % eval_interval == 0) or (epoch == stage2_epochs - 1) or SMOKE_TEST
        if should_eval:
            val_metrics = evaluate(
                model, val_loader, device, len(cls_num_list), cls_num_list=cls_num_list,
                criterion=criterion if isinstance(criterion, nn.Module) else None)
            row.update(scalar_metrics_for_log(val_metrics, prefix="val_"))
            score = float(val_metrics["selection_score"])
            if score > best_score:
                best_score = score
                best_val_metrics = val_metrics
                best_epoch = epoch + 1
                extra = {
                    "stage2_mode": stage2_mode,
                    "stage2_loss": loss_name,
                    "stage2_epochs": stage2_epochs,
                    "stage1_checkpoint": config.get("stage1_checkpoint"),
                    "reset_classifier": stage2_mode == "cRT_reset",
                    "maxnorm_radius": maxnorm_radius,
                    "optimizer_state_dict": optimizer.state_dict(),
                }
                save_training_checkpoint(
                    best_path, model, config, cls_num_list, epoch + 1,
                    val_metrics, stage_name="stage2", extra=extra, device=device)
                write_stage_metrics(run_dir, "val_metrics_stage2", val_metrics)
            set_backbone_eval_classifier_train(model)

        log_rows.append(row)
        write_csv(log_path, log_rows, fieldnames=epoch_log_fieldnames(log_rows))
        print(
            f"[Stage 2][{stage2_mode}] epoch {epoch + 1}/{stage2_epochs} "
            f"loss={train_loss:.4f} best={best_score:.3f}")

    if best_val_metrics is None:
        best_val_metrics = evaluate(model, val_loader, device, len(cls_num_list), cls_num_list=cls_num_list)
        best_score = float(best_val_metrics["selection_score"])
        best_epoch = stage2_epochs
        save_training_checkpoint(
            best_path, model, config, cls_num_list, best_epoch, best_val_metrics,
            stage_name="stage2", extra={"stage2_mode": stage2_mode, "stage2_loss": loss_name}, device=device)
        write_stage_metrics(run_dir, "val_metrics_stage2", best_val_metrics)

    load_training_checkpoint(model, best_path, device=device)
    model.eval()
    return {
        "model": model,
        "run_dir": str(run_dir),
        "best_checkpoint_path": str(best_path),
        "train_log_path": str(log_path),
        "best_val_metrics": best_val_metrics,
        "best_selection_score": best_score,
        "best_epoch": best_epoch,
        "stage2_mode": stage2_mode,
        "stage2_loss": loss_name,
    }


### Step 3 Smoke Helper

This helper is not run automatically. It builds one tiny split-trained run for a single scenario when the notebook runtime has the required packages and CIFAR data access.


In [ ]:
def run_step3_smoke_training(download=True):
    smoke_config = {
        "imb_type": "exp",
        "imb_factor": 0.1,
        "candidate_id": "step3_smoke",
        "model_name": "resnet32",
        "stage1_mode": "CE_StrongAug",
        "stage2_mode": "cRT_no_reset",
        "stage1_epochs": 1,
        "stage2_epochs": 1,
        "bn_align_epochs": 0,
        "eval_interval": 1,
        "stage2_eval_interval": 1,
        "use_ema": False,
        "overwrite_logs": True,
    }
    assert_training_budget(smoke_config)
    set_seed(DEFAULT_SEED)
    data = build_datasets_for_scenario("exp", 0.1, seed=DEFAULT_SEED, download=download)
    train_loader = make_natural_loader(data["train_dataset"], drop_last=True)
    balanced_loader = make_balanced_loader(data["train_dataset"], data["train_targets"], drop_last=True)
    val_loader = make_natural_loader(data["val_dataset"], shuffle=False)
    model = build_model("resnet32", NUM_CLASSES).to(DEVICE)
    stage1 = train_stage1_representation(
        smoke_config, train_loader, val_loader, model, data["train_cls_num_list"], DEVICE)
    smoke_config["stage1_checkpoint"] = stage1["best_checkpoint_path"]
    stage2 = train_stage2_classifier(
        smoke_config, stage1["model"], balanced_loader, val_loader,
        data["train_cls_num_list"], DEVICE)
    return {"stage1": stage1, "stage2": stage2}


## Calibration and Experiment Orchestration

Step 4 connects the data, model, training, and validation components into a bounded experiment runner. Search, checkpoint selection, calibration, and seed selection use validation only. The final test split remains guarded in the frozen evaluation cell.


In [ ]:
TAU_NORM_GRID = [0.0, 0.5] if SMOKE_TEST else [0.0, 0.25, 0.5, 0.75, 1.0]
LOGIT_ADJUST_GRID = [0.0, 0.5] if SMOKE_TEST else [0.0, 0.25, 0.5, 0.75, 1.0, 1.5]
T_HEAD_GRID = [1.0, 1.5] if SMOKE_TEST else [1.0, 1.25, 1.5, 2.0]
T_MEDIUM_GRID = [1.0] if SMOKE_TEST else [0.9, 1.0, 1.1, 1.25]
T_TAIL_GRID = [1.0] if SMOKE_TEST else [0.75, 1.0, 1.25]


def calibration_key(calibration):
    return (
        float(calibration.get("tau_norm", 0.0)),
        float(calibration.get("logit_adjust_tau", 0.0)),
        float(calibration.get("T_head", 1.0)),
        float(calibration.get("T_medium", 1.0)),
        float(calibration.get("T_tail", 1.0)),
    )


def calibration_candidates(grid=None):
    if grid is None:
        grid = {
            "tau_norm": TAU_NORM_GRID,
            "logit_adjust_tau": LOGIT_ADJUST_GRID,
            "T_head": T_HEAD_GRID,
            "T_medium": T_MEDIUM_GRID,
            "T_tail": T_TAIL_GRID,
        }
    seen = set()
    identity = default_calibration()
    candidates = [identity]
    seen.add(calibration_key(identity))
    for tau_norm in grid["tau_norm"]:
        for tau_la in grid["logit_adjust_tau"]:
            for t_head in grid["T_head"]:
                for t_medium in grid["T_medium"]:
                    for t_tail in grid["T_tail"]:
                        candidate = {
                            "tau_norm": float(tau_norm),
                            "logit_adjust_tau": float(tau_la),
                            "T_head": float(t_head),
                            "T_medium": float(t_medium),
                            "T_tail": float(t_tail),
                        }
                        key = calibration_key(candidate)
                        if key not in seen:
                            seen.add(key)
                            candidates.append(candidate)
    return candidates


def calibration_better(candidate_metrics, candidate_calibration, best_metrics, best_calibration):
    eps = 1e-9
    candidate_score = float(candidate_metrics["selection_score"])
    best_score = float(best_metrics["selection_score"])
    if candidate_score > best_score + eps:
        return True
    if abs(candidate_score - best_score) > eps:
        return False
    for key in ["overall_acc", "macro_acc", "tail_acc"]:
        cand = float(candidate_metrics.get(key, 0.0))
        best = float(best_metrics.get(key, 0.0))
        if cand > best + eps:
            return True
        if best > cand + eps:
            return False
    return calibration_strength(candidate_calibration) < calibration_strength(best_calibration)


def tune_calibration(model, val_loader, cls_num_list, device, groups=None,
                     run_dir=None, base_metrics=None, grid=None):
    if groups is None:
        groups = get_frequency_groups(cls_num_list)
    run_dir = Path(run_dir) if run_dir is not None else None
    classifier = model.get_classifier()
    original_state = clone_classifier_state(classifier)

    identity = default_calibration()
    if base_metrics is None:
        restore_classifier_state(classifier, original_state)
        base_metrics = evaluate(
            model, val_loader, device, len(cls_num_list), cls_num_list=cls_num_list,
            calibration=identity)

    best_calibration = dict(identity)
    best_metrics = base_metrics
    best_improved = False
    grid_rows = []

    for candidate in calibration_candidates(grid):
        restore_classifier_state(classifier, original_state)
        apply_tau_normalization_to_classifier(classifier, candidate.get("tau_norm", 0.0))
        metrics = evaluate(
            model, val_loader, device, len(cls_num_list), cls_num_list=cls_num_list,
            calibration=candidate)
        row = {
            "tau_norm": candidate["tau_norm"],
            "logit_adjust_tau": candidate["logit_adjust_tau"],
            "T_head": candidate["T_head"],
            "T_medium": candidate["T_medium"],
            "T_tail": candidate["T_tail"],
            **scalar_metrics_for_log(metrics, prefix="val_"),
            "selection_score": float(metrics["selection_score"]),
            "selected": False,
        }
        grid_rows.append(row)
        if calibration_better(metrics, candidate, best_metrics, best_calibration):
            best_calibration = dict(candidate)
            best_metrics = metrics
            best_improved = float(metrics["selection_score"]) > float(base_metrics["selection_score"])

    if not best_improved:
        best_calibration = dict(identity)
        best_metrics = base_metrics

    for row in grid_rows:
        row["selected"] = calibration_key(row) == calibration_key(best_calibration)

    restore_classifier_state(classifier, original_state)
    if run_dir is not None:
        write_csv(run_dir / "calibration_grid.csv", grid_rows)
        write_json(run_dir / "calibration_selected.json", {
            "selected_calibration": best_calibration,
            "best_metrics": metrics_without_large_arrays(best_metrics),
            "base_metrics": metrics_without_large_arrays(base_metrics),
            "improved_over_identity": bool(best_improved),
        })

    return {
        "selected_calibration": best_calibration,
        "best_metrics": best_metrics,
        "base_metrics": base_metrics,
        "grid_rows": grid_rows,
        "improved_over_identity": bool(best_improved),
    }


def apply_selected_calibration_to_model(model, calibration):
    classifier = model.get_classifier()
    apply_tau_normalization_to_classifier(classifier, calibration.get("tau_norm", 0.0))
    return model


### Candidate Policy

The search is intentionally bounded. Smoke mode runs a tiny configuration. Full mode starts with `resnet32`, searches Stage 1 variants, keeps the top Stage 1 checkpoints, and only then tries Stage 2 variants and calibration.


In [ ]:
def base_candidate_config(imb_type, imb_factor, seed=0, model_name="resnet32"):
    return {
        "imb_type": imb_type,
        "imb_factor": imb_factor,
        "seed": int(seed),
        "model_name": model_name,
        "stage1_epochs": STAGE1_EPOCHS,
        "stage2_epochs": STAGE2_EPOCHS,
        "bn_align_epochs": BN_ALIGN_EPOCHS,
        "warmup_epochs": WARMUP_EPOCHS,
        "drw_start_epoch": DRW_START_EPOCH,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "final_track": FINAL_SUBMISSION_TRACK,
        "download": True,
    }


def stage1_candidate_grid(imb_type, imb_factor, seed=0, model_name="resnet32"):
    modes = ["CE_StrongAug", "LDAM_DRW"] if SMOKE_TEST else ["CE_StrongAug", "LDAM_DRW", "BalancedSoftmax"]
    candidates = []
    for mode in modes:
        cfg = base_candidate_config(imb_type, imb_factor, seed=seed, model_name=model_name)
        cfg.update({
            "stage1_mode": mode,
            "stage2_mode": "none",
            "stop_after_stage1": True,
            "tune_calibration": False,
            "candidate_id": f"stage1_{model_name}_{mode}_seed{seed}",
            "use_ema": not SMOKE_TEST,
        })
        if mode == "BalancedSoftmax":
            cfg.update({"mixup_alpha": 0.0, "cutmix_alpha": 0.0})
        candidates.append(cfg)
    return candidates


def stage2_candidate_grid(stage1_result):
    base = dict(stage1_result["config"])
    base.pop("run_dir", None)
    base.pop("output_dir", None)
    base["stop_after_stage1"] = False
    base["skip_stage1"] = True
    base["stage1_checkpoint"] = stage1_result["stage1_checkpoint_path"]
    base["tune_calibration"] = True

    modes = ["none", "cRT_no_reset"] if SMOKE_TEST else [
        "none",
        "cRT_no_reset",
        "cRT_reset",
        "cRT_no_reset_LabelAwareSmoothing",
    ]
    configs = []
    for mode in modes:
        cfg = dict(base)
        cfg["stage2_mode"] = mode
        cfg["stage2_epochs"] = 0 if mode == "none" else STAGE2_EPOCHS
        cfg["candidate_id"] = (
            f"talc_{cfg['model_name']}_{cfg['stage1_mode']}_{mode}_seed{cfg['seed']}"
        )
        if mode == "cRT_no_reset_LabelAwareSmoothing":
            cfg.update({"stage2_loss": "LabelAwareSmoothingCE", "eps_max": 0.2})
        else:
            cfg.setdefault("stage2_loss", "CE")
        configs.append(cfg)
    return configs


def result_metric_row(result, selected=False):
    metrics = result.get("calibrated_val_metrics") or result.get("stage2_val_metrics") or result.get("stage1_val_metrics")
    config = result["config"]
    size_info = result.get("size_info", {})
    calibration = result.get("calibration", default_calibration())
    return {
        "scenario": scenario_tag(config["imb_type"], config["imb_factor"]),
        "candidate_id": config.get("candidate_id", ""),
        "model": config.get("model_name", ""),
        "stage1_mode": config.get("stage1_mode", ""),
        "stage2_mode": config.get("stage2_mode", ""),
        "bn_align_mode": config.get("bn_align_mode", "none"),
        "calibration": json.dumps(to_serializable(calibration), sort_keys=True),
        "val_overall_acc": float(metrics.get("overall_acc", 0.0)),
        "val_macro_acc": float(metrics.get("macro_acc", 0.0)),
        "val_tail_acc": float(metrics.get("tail_acc", 0.0)),
        "selection_score": float(metrics.get("selection_score", 0.0)),
        "val_class_coverage": float(metrics.get("val_class_coverage", 0.0)),
        "tail_val_coverage": float(metrics.get("tail_val_coverage", 0.0)),
        "params_M": size_info.get("params_M"),
        "FLOPs_M": size_info.get("FLOPs_M"),
        "selected": bool(selected),
    }


def write_per_class_validation_outputs(run_dir, metrics, class_names, train_cls_num_list, val_cls_num_list, groups):
    run_dir = Path(run_dir)
    per_class_acc = np.asarray(metrics["per_class_acc"])
    per_class_count = np.asarray(metrics["per_class_count"])
    group_per_class = groups["group_per_class"]
    rows = []
    for class_id in range(len(per_class_count)):
        rows.append({
            "class_id": class_id,
            "class_name": class_names[class_id] if class_id < len(class_names) else str(class_id),
            "train_count": int(train_cls_num_list[class_id]),
            "val_count": int(val_cls_num_list[class_id]),
            "eval_count": int(per_class_count[class_id]),
            "per_class_acc": None if np.isnan(per_class_acc[class_id]) else float(per_class_acc[class_id]),
            "group": str(group_per_class[class_id]),
        })
    write_csv(run_dir / "per_class_val.csv", rows)
    np.save(run_dir / "confusion_val.npy", metrics["confusion"])


### Single Candidate Runner

`run_single_candidate` does not touch the test split unless `final_test=True`. Normal candidate search uses training-derived validation only.


In [ ]:
def run_single_candidate(config, final_test=False):
    config = dict(config or {})
    assert not final_test, "Use run_final_frozen_evaluation for test evaluation after configs are frozen."
    seed = int(config.get("seed", DEFAULT_SEED))
    set_seed(seed)
    assert_training_budget(config)

    imb_type = config["imb_type"]
    imb_factor = config["imb_factor"]
    candidate_id = config.get("candidate_id", f"candidate_seed{seed}")
    run_dir = candidate_dir(imb_type, imb_factor, candidate_id)
    config["run_dir"] = str(run_dir)
    write_json(run_dir / "config.json", config)

    data = build_datasets_for_scenario(
        imb_type, imb_factor, seed=seed, download=bool(config.get("download", True)),
        use_randaugment=bool(config.get("use_randaugment", True)),
        use_cutout=bool(config.get("use_cutout", True)),
        use_random_erasing=bool(config.get("use_random_erasing", False)),
    )
    train_loader = make_natural_loader(data["train_dataset"], shuffle=True, drop_last=True)
    train_balanced_loader = make_balanced_loader(
        data["train_dataset"], data["train_targets"], drop_last=True)
    val_loader = make_natural_loader(data["val_dataset"], shuffle=False, drop_last=False)
    cls_num_list = data["train_cls_num_list"]
    groups = get_frequency_groups(cls_num_list)

    model = build_model(config.get("model_name", "resnet32"), len(cls_num_list)).to(DEVICE)
    stage1_result = None

    if config.get("skip_stage1", False):
        checkpoint_path = config.get("stage1_checkpoint")
        assert checkpoint_path, "skip_stage1=True requires stage1_checkpoint."
        load_training_checkpoint(model, checkpoint_path, device=DEVICE)
        stage1_metrics = evaluate(model, val_loader, DEVICE, len(cls_num_list), cls_num_list=cls_num_list)
        stage1_result = {
            "model": model,
            "run_dir": str(run_dir),
            "best_checkpoint_path": checkpoint_path,
            "best_val_metrics": stage1_metrics,
            "best_selection_score": float(stage1_metrics["selection_score"]),
            "best_epoch": None,
            "best_source": "loaded",
            "use_ema_stage1": bool(config.get("use_ema_stage1", False)),
        }
    else:
        stage1_result = train_stage1_representation(
            config, train_loader, val_loader, model, cls_num_list, DEVICE)
        model = stage1_result["model"]

    size_info = safe_model_size_summary(model, DEVICE)

    if config.get("stop_after_stage1", False):
        metrics = stage1_result["best_val_metrics"]
        write_per_class_validation_outputs(
            run_dir, metrics, data["class_names"], cls_num_list, data["val_cls_num_list"], groups)
        result = {
            "config": dict(config),
            "run_dir": str(run_dir),
            "stage1_checkpoint_path": stage1_result["best_checkpoint_path"],
            "stage1_val_metrics": metrics,
            "size_info": size_info,
            "train_cls_num_list": cls_num_list,
            "val_cls_num_list": data["val_cls_num_list"],
            "class_names": data["class_names"],
            "calibration": default_calibration(),
        }
        write_json(run_dir / "candidate_result.json", {
            **{k: v for k, v in result.items() if k not in {"stage1_val_metrics"}},
            "stage1_val_metrics": metrics_without_large_arrays(metrics),
        })
        return result

    config["stage1_checkpoint"] = stage1_result["best_checkpoint_path"]
    stage2_result = train_stage2_classifier(
        config, model, train_balanced_loader, val_loader, cls_num_list, DEVICE)
    model = stage2_result["model"]
    stage2_metrics = stage2_result["best_val_metrics"]
    calibration = default_calibration()
    calibrated_metrics = stage2_metrics
    calibration_result = {
        "selected_calibration": calibration,
        "best_metrics": calibrated_metrics,
        "base_metrics": stage2_metrics,
        "grid_rows": [],
        "improved_over_identity": False,
    }

    if bool(config.get("tune_calibration", True)):
        calibration_result = tune_calibration(
            model, val_loader, cls_num_list, DEVICE, groups=groups,
            run_dir=run_dir, base_metrics=stage2_metrics)
        calibration = calibration_result["selected_calibration"]
        calibrated_metrics = calibration_result["best_metrics"]
        apply_selected_calibration_to_model(model, calibration)

    best_calibrated_path = run_dir / "best_calibrated.pth"
    save_training_checkpoint(
        best_calibrated_path, model, config, cls_num_list,
        stage2_result.get("best_epoch") or 0, calibrated_metrics,
        stage_name="calibrated", extra={
            "calibration": calibration,
            "stage1_checkpoint": stage1_result["best_checkpoint_path"],
            "stage2_checkpoint": stage2_result["best_checkpoint_path"],
            "use_ema_stage1": bool(stage1_result.get("use_ema_stage1", False)),
            "bn_align_mode": config.get("bn_align_mode", "none"),
        }, device=DEVICE)

    write_per_class_validation_outputs(
        run_dir, calibrated_metrics, data["class_names"], cls_num_list, data["val_cls_num_list"], groups)

    result = {
        "config": dict(config),
        "run_dir": str(run_dir),
        "stage1_checkpoint_path": stage1_result["best_checkpoint_path"],
        "stage2_checkpoint_path": stage2_result["best_checkpoint_path"],
        "calibrated_checkpoint_path": str(best_calibrated_path),
        "stage1_val_metrics": stage1_result["best_val_metrics"],
        "stage2_val_metrics": stage2_metrics,
        "calibrated_val_metrics": calibrated_metrics,
        "calibration": calibration,
        "calibration_improved": bool(calibration_result.get("improved_over_identity", False)),
        "size_info": safe_model_size_summary(model, DEVICE),
        "train_cls_num_list": cls_num_list,
        "val_cls_num_list": data["val_cls_num_list"],
        "class_names": data["class_names"],
        "final_checkpoint_path": str(best_calibrated_path),
    }
    result["config"].update({
        "calibration": calibration,
        "final_checkpoint_path": str(best_calibrated_path),
        "train_cls_num_list": cls_num_list,
        "val_cls_num_list": data["val_cls_num_list"],
        "class_names": data["class_names"],
        "use_ema_stage1": bool(stage1_result.get("use_ema_stage1", False)),
    })
    write_json(run_dir / "candidate_result.json", {
        **{k: v for k, v in result.items() if k not in {"stage1_val_metrics", "stage2_val_metrics", "calibrated_val_metrics"}},
        "stage1_val_metrics": metrics_without_large_arrays(stage1_result["best_val_metrics"]),
        "stage2_val_metrics": metrics_without_large_arrays(stage2_metrics),
        "calibrated_val_metrics": metrics_without_large_arrays(calibrated_metrics),
    })
    return result


### Scenario Search and Seed Confirmation

Scenario search writes validation summaries and frozen selected configs. Seed confirmation reruns only the selected method and chooses the final seed by validation score.


In [ ]:
def result_sort_key(result):
    metrics = result.get("calibrated_val_metrics") or result.get("stage2_val_metrics") or result.get("stage1_val_metrics")
    size_info = result.get("size_info", {})
    flops = size_info.get("FLOPs_M")
    params = size_info.get("params_M")
    return (
        float(metrics.get("selection_score", 0.0)),
        float(metrics.get("overall_acc", 0.0)),
        float(metrics.get("macro_acc", 0.0)),
        float(metrics.get("tail_acc", 0.0)),
        -float(flops) if flops is not None else 0.0,
        -float(params) if params is not None else 0.0,
        -calibration_strength(result.get("calibration", default_calibration())),
    )


def _read_csv_rows(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open("r", newline="") as f:
        return list(csv.DictReader(f))


def merge_global_rows(path, new_rows, scenario):
    path = Path(path)
    existing = [row for row in _read_csv_rows(path) if row.get("scenario") != scenario]
    merged = existing + list(new_rows)
    write_csv(path, merged)
    return merged


def upsert_selected_config(selected_config):
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    path = RESULTS_DIR / "selected_configs.json"
    if path.exists():
        existing = read_json(path)
    else:
        existing = {}
    if isinstance(existing, list):
        converted = {}
        for item in existing:
            key = scenario_tag(item["imb_type"], item["imb_factor"])
            converted[key] = item
        existing = converted
    key = scenario_tag(selected_config["imb_type"], selected_config["imb_factor"])
    existing[key] = to_serializable(selected_config)
    write_json(path, existing)
    return path


def run_scenario_model_search(imb_type, imb_factor):
    seed = SEARCH_SEEDS[0]
    all_results = []
    stage1_results = []
    for model_name in MODEL_CANDIDATES:
        for cfg in stage1_candidate_grid(imb_type, imb_factor, seed=seed, model_name=model_name):
            result = run_single_candidate(cfg, final_test=False)
            stage1_results.append(result)
            all_results.append(result)

    top_k = 1 if SMOKE_TEST else 2
    top_stage1 = sorted(stage1_results, key=result_sort_key, reverse=True)[:top_k]
    stage2_results = []
    for stage1_result in top_stage1:
        for cfg in stage2_candidate_grid(stage1_result):
            result = run_single_candidate(cfg, final_test=False)
            stage2_results.append(result)
            all_results.append(result)

    selectable = stage2_results if stage2_results else stage1_results
    selected = sorted(selectable, key=result_sort_key, reverse=True)[0]
    selected_config = dict(selected["config"])
    selected_config.update({
        "selected": True,
        "selected_by": "validation_selection_score",
        "scenario": scenario_tag(imb_type, imb_factor),
        "final_checkpoint_path": selected.get("final_checkpoint_path") or selected.get("stage1_checkpoint_path"),
        "calibration": selected.get("calibration", default_calibration()),
        "selection_score": float((selected.get("calibrated_val_metrics") or selected.get("stage2_val_metrics") or selected.get("stage1_val_metrics"))["selection_score"]),
        "train_cls_num_list": selected.get("train_cls_num_list"),
        "val_cls_num_list": selected.get("val_cls_num_list"),
        "class_names": selected.get("class_names"),
    })

    scenario = scenario_tag(imb_type, imb_factor)
    rows = [result_metric_row(result, selected=(result is selected)) for result in all_results]
    all_candidate_rows = merge_global_rows(RESULTS_DIR / "validation_candidate_summary.csv", rows, scenario)
    pareto_rows = sorted(all_candidate_rows, key=lambda row: float(row.get("selection_score", 0.0)), reverse=True)
    write_csv(RESULTS_DIR / "validation_pareto_summary.csv", pareto_rows)
    upsert_selected_config(selected_config)

    calibration_rows = []
    for result in all_results:
        grid_path = Path(result["run_dir"]) / "calibration_grid.csv"
        if grid_path.exists():
            for row in _read_csv_rows(grid_path):
                row.update({
                    "scenario": scenario_tag(imb_type, imb_factor),
                    "candidate_id": result["config"].get("candidate_id", ""),
                })
                calibration_rows.append(row)
    if calibration_rows:
        merge_global_rows(RESULTS_DIR / "calibration_summary.csv", calibration_rows, scenario)

    write_json(selected_dir(imb_type, imb_factor) / "selected_config.json", selected_config)
    return {
        "selected_config": selected_config,
        "selected_result": selected,
        "all_results": all_results,
        "summary_rows": rows,
    }


def run_all_scenario_model_searches():
    results = []
    for imb_type, imb_factor in SCENARIOS:
        results.append(run_scenario_model_search(imb_type, imb_factor))
    return results


def run_seed_confirmation(selected_config):
    rows = []
    seed_results = []
    for seed in CONFIRM_SEEDS:
        cfg = dict(selected_config)
        cfg["seed"] = int(seed)
        cfg["candidate_id"] = f"seed_confirm_{scenario_tag(cfg['imb_type'], cfg['imb_factor'])}_seed{seed}"
        cfg.pop("run_dir", None)
        cfg.pop("stage1_checkpoint", None)
        cfg.pop("final_checkpoint_path", None)
        cfg["skip_stage1"] = False
        cfg["stop_after_stage1"] = False
        result = run_single_candidate(cfg, final_test=False)
        seed_results.append(result)
        metrics = result.get("calibrated_val_metrics") or result.get("stage2_val_metrics")
        rows.append({
            "scenario": scenario_tag(cfg["imb_type"], cfg["imb_factor"]),
            "method": "TALC++",
            "seed": int(seed),
            "val_overall_acc": float(metrics["overall_acc"]),
            "val_macro_acc": float(metrics["macro_acc"]),
            "val_tail_acc": float(metrics["tail_acc"]),
            "selection_score": float(metrics["selection_score"]),
            "selected_final_seed": False,
            "final_checkpoint_path": result.get("final_checkpoint_path"),
        })

    selected_result = sorted(seed_results, key=result_sort_key, reverse=True)[0]
    selected_seed = int(selected_result["config"]["seed"])
    for row in rows:
        row["selected_final_seed"] = row["seed"] == selected_seed

    selected_final_config = dict(selected_result["config"])
    selected_final_config.update({
        "selected_final_seed": selected_seed,
        "final_checkpoint_path": selected_result.get("final_checkpoint_path"),
        "calibration": selected_result.get("calibration", default_calibration()),
        "selection_score": float((selected_result.get("calibrated_val_metrics") or selected_result.get("stage2_val_metrics"))["selection_score"]),
    })
    write_csv(RESULTS_DIR / "seed_confirm_summary.csv", rows)
    upsert_selected_config(selected_final_config)
    return {
        "selected_config": selected_final_config,
        "seed_results": seed_results,
        "rows": rows,
    }


## FINAL FROZEN TEST EVALUATION - DO NOT RUN UNTIL CONFIGS ARE FIXED

The test split must only be used here after selected configs have been written to `results/selected_configs.json`. This cell is guarded by `RUN_FINAL_TEST`.


In [ ]:
def _selected_configs_iter(selected_configs):
    if isinstance(selected_configs, dict):
        if all(isinstance(v, dict) for v in selected_configs.values()):
            return list(selected_configs.values())
        return [selected_configs]
    return list(selected_configs)


def test_summary_row(config, metrics, size_info):
    calibration = config.get("calibration", default_calibration())
    return {
        "method": config.get("method", "TALC++"),
        "model": config.get("model_name", "resnet32"),
        "imb_type": config["imb_type"],
        "imb_factor": config["imb_factor"],
        "seed": config.get("selected_final_seed", config.get("seed", DEFAULT_SEED)),
        "overall_acc": float(metrics.get("overall_acc", 0.0)),
        "macro_acc": float(metrics.get("macro_acc", 0.0)),
        "head_acc": float(metrics.get("head_acc", 0.0)),
        "medium_acc": float(metrics.get("medium_acc", 0.0)),
        "tail_acc": float(metrics.get("tail_acc", 0.0)),
        "head50_acc": float(metrics.get("head50_acc", 0.0)),
        "tail50_acc": float(metrics.get("tail50_acc", 0.0)),
        "params_M": size_info.get("params_M"),
        "trainable_params_M": size_info.get("trainable_params_M"),
        "MACs_M": size_info.get("MACs_M"),
        "FLOPs_M": size_info.get("FLOPs_M"),
        "stage1_mode": config.get("stage1_mode", ""),
        "stage2_mode": config.get("stage2_mode", ""),
        "bn_align_mode": config.get("bn_align_mode", "none"),
        "tau_norm": calibration.get("tau_norm", 0.0),
        "logit_adjust_tau": calibration.get("logit_adjust_tau", 0.0),
        "T_head": calibration.get("T_head", 1.0),
        "T_medium": calibration.get("T_medium", 1.0),
        "T_tail": calibration.get("T_tail", 1.0),
        "use_ema_stage1": config.get("use_ema_stage1", False),
        "final_track": config.get("final_track", FINAL_SUBMISSION_TRACK),
    }


def write_per_class_test_outputs(out_dir, metrics, class_names, train_cls_num_list, test_cls_num_list, groups):
    out_dir = Path(out_dir)
    per_class_acc = np.asarray(metrics["per_class_acc"])
    per_class_count = np.asarray(metrics["per_class_count"])
    group_per_class = groups["group_per_class"]
    rows = []
    for class_id in range(len(per_class_count)):
        rows.append({
            "class_id": class_id,
            "class_name": class_names[class_id] if class_id < len(class_names) else str(class_id),
            "train_count": int(train_cls_num_list[class_id]),
            "test_count": int(test_cls_num_list[class_id]),
            "eval_count": int(per_class_count[class_id]),
            "per_class_acc": None if np.isnan(per_class_acc[class_id]) else float(per_class_acc[class_id]),
            "group": str(group_per_class[class_id]),
        })
    write_csv(out_dir / "per_class_test.csv", rows)
    np.save(out_dir / "confusion_test.npy", metrics["confusion"])


def run_final_frozen_evaluation(selected_configs):
    assert RUN_FINAL_TEST, "Final test evaluation is guarded. Set RUN_FINAL_TEST=True only after configs are frozen."
    rows = []
    for config in _selected_configs_iter(selected_configs):
        config = dict(config)
        checkpoint_path = config.get("final_checkpoint_path")
        assert checkpoint_path, "Each selected config must include final_checkpoint_path."
        assert Path(checkpoint_path).exists(), f"Missing final checkpoint: {checkpoint_path}"

        seed = int(config.get("selected_final_seed", config.get("seed", DEFAULT_SEED)))
        set_seed(seed)
        data = build_datasets_for_scenario(
            config["imb_type"], config["imb_factor"], seed=seed,
            download=bool(config.get("download", True)),
            use_randaugment=False, use_cutout=False, use_random_erasing=False)
        test_loader = make_natural_loader(data["test_dataset"], batch_size=100, shuffle=False, drop_last=False)
        cls_num_list = config.get("train_cls_num_list") or data["train_cls_num_list"]

        model = build_model(config.get("model_name", "resnet32"), len(cls_num_list)).to(DEVICE)
        checkpoint = load_training_checkpoint(model, checkpoint_path, device=DEVICE)
        calibration = config.get("calibration") or checkpoint.get("calibration", default_calibration())
        metrics = evaluate(
            model, test_loader, DEVICE, len(cls_num_list), cls_num_list=cls_num_list,
            calibration=calibration, return_details=True)
        size_info = safe_model_size_summary(model, DEVICE)

        out_dir = selected_dir(config["imb_type"], config["imb_factor"])
        write_json(out_dir / "test_metrics.json", metrics_without_large_arrays(metrics))
        write_per_class_test_outputs(
            out_dir, metrics, data["class_names"], cls_num_list,
            count_classes(data["test_targets"], len(cls_num_list)).tolist(),
            get_frequency_groups(cls_num_list))
        rows.append(test_summary_row(config, metrics, size_info))

    write_csv(RESULTS_DIR / "final_test_summary.csv", rows)
    write_json(RESULTS_DIR / "final_test_summary.json", rows)
    return rows


if RUN_FINAL_TEST:
    selected_configs_path = RESULTS_DIR / "selected_configs.json"
    assert selected_configs_path.exists(), "Freeze selected configs before final test evaluation."
    selected_configs = read_json(selected_configs_path)
    final_results = run_final_frozen_evaluation(selected_configs)
    with (ROOT_DIR / "FROZEN_RESULTS_DO_NOT_TUNE.txt").open("w") as f:
        f.write("Final frozen test evaluation has been run. Do not tune configs after this point.\n")
else:
    print("Final frozen test evaluation is disabled. Set RUN_FINAL_TEST=True only after configs are fixed.")


## Results, Figures, and Report Notes

These helpers turn saved validation/test artifacts into report-ready CSV files, diagnostic figures, final weight copies, and `report_notes.md`. They are designed to be safe before results exist: missing inputs create empty summaries or short placeholder notes instead of crashing.


In [ ]:
FINAL_TABLE_COLUMNS = [
    "method", "model", "imb_type", "imb_factor", "seed",
    "overall_acc", "macro_acc", "head_acc", "medium_acc", "tail_acc", "head50_acc", "tail50_acc",
    "params_M", "trainable_params_M", "MACs_M", "FLOPs_M",
    "stage1_mode", "stage2_mode", "bn_align_mode",
    "tau_norm", "logit_adjust_tau", "T_head", "T_medium", "T_tail",
    "use_ema_stage1", "final_track",
]


def read_csv_rows(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open("r", newline="") as f:
        return list(csv.DictReader(f))


def maybe_float(value, default=0.0):
    if value in (None, "", "None"):
        return default
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def maybe_int(value, default=0):
    if value in (None, "", "None"):
        return default
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return default


def scenario_from_row(row):
    if row.get("scenario"):
        return row["scenario"]
    if row.get("imb_type") and row.get("imb_factor"):
        return scenario_tag(row["imb_type"], row["imb_factor"])
    return "unknown"


def method_name_from_row(row):
    stage1 = row.get("stage1_mode", "")
    stage2 = row.get("stage2_mode", "")
    calibration_text = row.get("calibration", "")
    if stage2 in ("", "none") and stage1 == "CE_StrongAug":
        return "CE_StrongAug"
    if stage2 in ("", "none") and stage1 == "LDAM_DRW":
        return "LDAM_DRW"
    if stage2 in ("", "none") and stage1 == "BalancedSoftmax":
        return "BalancedSoftmax"
    if calibration_text and calibration_text not in ("{}", ""):
        try:
            calibration = json.loads(calibration_text)
        except Exception:
            calibration = default_calibration()
        if calibration_strength(calibration) > 0.0:
            return "TALC++"
    return "TALC++_no_calib"


def normalize_final_summary_rows(rows):
    normalized = []
    for row in rows:
        out = {key: row.get(key, "") for key in FINAL_TABLE_COLUMNS}
        out["method"] = out["method"] or method_name_from_row(row)
        out["model"] = out["model"] or row.get("model_name", row.get("model", "resnet32"))
        out["seed"] = maybe_int(out["seed"], maybe_int(row.get("selected_final_seed"), maybe_int(row.get("seed"), DEFAULT_SEED)))
        for key in [
            "overall_acc", "macro_acc", "head_acc", "medium_acc", "tail_acc", "head50_acc", "tail50_acc",
            "params_M", "trainable_params_M", "MACs_M", "FLOPs_M",
            "tau_norm", "logit_adjust_tau", "T_head", "T_medium", "T_tail",
        ]:
            out[key] = maybe_float(out[key], None)
        out["bn_align_mode"] = out["bn_align_mode"] or "none"
        out["final_track"] = out["final_track"] or FINAL_SUBMISSION_TRACK
        normalized.append(out)
    return normalized


def refresh_final_test_summary():
    rows = normalize_final_summary_rows(read_csv_rows(RESULTS_DIR / "final_test_summary.csv"))
    write_csv(RESULTS_DIR / "final_test_summary.csv", rows, fieldnames=FINAL_TABLE_COLUMNS)
    return rows


def generate_ablation_summary():
    source_rows = read_csv_rows(RESULTS_DIR / "final_test_summary.csv")
    metric_prefix = ""
    if not source_rows:
        source_rows = read_csv_rows(RESULTS_DIR / "validation_candidate_summary.csv")
        metric_prefix = "val_"
    ablation_rows = []
    by_scenario = defaultdict(list)
    for row in source_rows:
        by_scenario[scenario_from_row(row)].append(row)

    for scenario, rows in by_scenario.items():
        baseline = None
        for row in rows:
            method = row.get("method") or method_name_from_row(row)
            if method in {"CE", "CE_StrongAug"} or row.get("stage1_mode") == "CE_StrongAug":
                baseline = row
                break
        base_overall = maybe_float((baseline or {}).get(metric_prefix + "overall_acc", (baseline or {}).get("overall_acc")))
        base_tail = maybe_float((baseline or {}).get(metric_prefix + "tail_acc", (baseline or {}).get("tail_acc")))
        for row in rows:
            method = row.get("method") or method_name_from_row(row)
            overall = maybe_float(row.get(metric_prefix + "overall_acc", row.get("overall_acc")))
            tail = maybe_float(row.get(metric_prefix + "tail_acc", row.get("tail_acc")))
            ablation_rows.append({
                "scenario": scenario,
                "method": method,
                "overall_acc": overall,
                "macro_acc": maybe_float(row.get(metric_prefix + "macro_acc", row.get("macro_acc"))),
                "head_acc": maybe_float(row.get(metric_prefix + "head_acc", row.get("head_acc"))),
                "medium_acc": maybe_float(row.get(metric_prefix + "medium_acc", row.get("medium_acc"))),
                "tail_acc": tail,
                "params_M": maybe_float(row.get("params_M"), None),
                "FLOPs_M": maybe_float(row.get("FLOPs_M"), None),
                "delta_over_CE": overall - base_overall if baseline is not None else None,
                "delta_tail_over_CE": tail - base_tail if baseline is not None else None,
            })
    write_csv(RESULTS_DIR / "ablation_summary.csv", ablation_rows)
    return ablation_rows


def collect_per_class_all_scenarios():
    rows = []
    for imb_type, imb_factor in SCENARIOS:
        scenario = scenario_tag(imb_type, imb_factor)
        selected_path = selected_dir(imb_type, imb_factor) / "per_class_test.csv"
        if not selected_path.exists():
            selected_path = selected_dir(imb_type, imb_factor) / "per_class_val.csv"
        for row in read_csv_rows(selected_path):
            row = dict(row)
            row["scenario"] = scenario
            rows.append(row)
    write_csv(RESULTS_DIR / "per_class_all_scenarios.csv", rows)
    return rows


def ensure_final_weight_artifacts(selected_configs=None):
    import shutil
    if selected_configs is None:
        path = RESULTS_DIR / "selected_configs.json"
        selected_configs = read_json(path) if path.exists() else {}
    copied = []
    for config in _selected_configs_iter(selected_configs):
        checkpoint_path = config.get("final_checkpoint_path")
        if not checkpoint_path:
            continue
        src = Path(checkpoint_path)
        if not src.exists():
            continue
        dst = selected_dir(config["imb_type"], config["imb_factor"]) / "final_model.pth"
        dst.parent.mkdir(parents=True, exist_ok=True)
        if src.resolve() != dst.resolve():
            shutil.copy2(src, dst)
        metadata_path = dst.with_suffix(".metadata.json")
        write_json(metadata_path, config)
        config["final_model_path"] = str(dst)
        copied.append({
            "scenario": scenario_tag(config["imb_type"], config["imb_factor"]),
            "source": str(src),
            "final_model_path": str(dst),
            "metadata_path": str(metadata_path),
        })
    if copied:
        write_csv(RESULTS_DIR / "final_weight_artifacts.csv", copied)
    return copied


def generate_result_tables():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    final_rows = refresh_final_test_summary() if (RESULTS_DIR / "final_test_summary.csv").exists() else []
    ablation_rows = generate_ablation_summary()
    per_class_rows = collect_per_class_all_scenarios()
    if not (RESULTS_DIR / "seed_confirm_summary.csv").exists():
        write_csv(RESULTS_DIR / "seed_confirm_summary.csv", [])
    if not (RESULTS_DIR / "calibration_summary.csv").exists():
        write_csv(RESULTS_DIR / "calibration_summary.csv", [])
    copied_weights = ensure_final_weight_artifacts()
    summary = {
        "final_test_rows": len(final_rows),
        "ablation_rows": len(ablation_rows),
        "per_class_rows": len(per_class_rows),
        "copied_final_weights": len(copied_weights),
    }
    write_json(RESULTS_DIR / "report_artifact_summary.json", summary)
    return summary


### Diagnostic and Qualitative Figures

Figure functions use matplotlib when available and otherwise write a small `.txt` placeholder so report generation does not fail in minimal environments.


In [ ]:
def get_matplotlib_pyplot():
    try:
        import matplotlib.pyplot as plt
        return plt
    except Exception:
        return None


def write_figure_placeholder(path, reason):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    txt_path = path.with_suffix(path.suffix + ".txt")
    txt_path.write_text(f"Figure not generated: {reason}\n")
    return str(txt_path)


def save_or_placeholder(fig, path, reason=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if fig is None:
        return write_figure_placeholder(path, reason or "matplotlib unavailable")
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    get_matplotlib_pyplot().close(fig)
    return str(path)


def plot_class_distribution(cls_num_list, groups, out_path):
    plt = get_matplotlib_pyplot()
    if plt is None:
        return write_figure_placeholder(out_path, "matplotlib unavailable")
    counts = np.asarray(cls_num_list)
    order = groups["order"]
    colors = []
    group_per_class = groups["group_per_class"]
    palette = {"head": "#4C78A8", "medium": "#F58518", "tail": "#54A24B"}
    for cls in order:
        colors.append(palette.get(str(group_per_class[cls]), "#999999"))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(np.arange(len(order)), counts[order], color=colors, width=0.9)
    ax.set_title("Training class distribution")
    ax.set_xlabel("Classes sorted by frequency")
    ax.set_ylabel("Training count")
    ax.set_yscale("log")
    return save_or_placeholder(fig, out_path)


def plot_per_class_acc_vs_freq(per_class_acc, train_counts, groups, out_path):
    plt = get_matplotlib_pyplot()
    if plt is None:
        return write_figure_placeholder(out_path, "matplotlib unavailable")
    acc = np.asarray(per_class_acc, dtype=np.float64)
    counts = np.asarray(train_counts, dtype=np.float64)
    group_per_class = groups["group_per_class"]
    fig, ax = plt.subplots(figsize=(6, 4))
    for group, color in [("head", "#4C78A8"), ("medium", "#F58518"), ("tail", "#54A24B")]:
        idx = np.where(group_per_class == group)[0]
        idx = idx[~np.isnan(acc[idx])]
        if len(idx) > 0:
            ax.scatter(counts[idx], acc[idx], label=group, s=26, alpha=0.8, color=color)
    ax.set_xscale("log")
    ax.set_xlabel("Training count")
    ax.set_ylabel("Per-class accuracy (%)")
    ax.set_title("Per-class accuracy vs. frequency")
    ax.legend()
    return save_or_placeholder(fig, out_path)


def tail_to_head_confusions(confusion, groups, class_names, top_k=20):
    confusion = np.asarray(confusion)
    head = set(map(int, groups["head"]))
    tail = list(map(int, groups["tail"]))
    rows = []
    for true_cls in tail:
        for pred_cls in head:
            count = int(confusion[true_cls, pred_cls])
            if count <= 0:
                continue
            rows.append({
                "true_class": true_cls,
                "true_name": class_names[true_cls] if true_cls < len(class_names) else str(true_cls),
                "pred_class": pred_cls,
                "pred_name": class_names[pred_cls] if pred_cls < len(class_names) else str(pred_cls),
                "count": count,
            })
    rows.sort(key=lambda row: row["count"], reverse=True)
    return rows[:top_k]


def plot_tail_to_head_confusion(confusion, groups, class_names, out_dir, top_k=20):
    out_dir = Path(out_dir)
    rows = tail_to_head_confusions(confusion, groups, class_names, top_k=top_k)
    write_csv(out_dir / "tail_to_head_confusions.csv", rows)
    plt = get_matplotlib_pyplot()
    out_path = out_dir / "confusion_tail_to_head.png"
    if plt is None:
        return write_figure_placeholder(out_path, "matplotlib unavailable")
    if not rows:
        return write_figure_placeholder(out_path, "no tail-to-head confusions available")
    labels = [f"{r['true_class']}->{r['pred_class']}" for r in rows]
    values = [r["count"] for r in rows]
    fig, ax = plt.subplots(figsize=(max(6, len(rows) * 0.35), 4))
    ax.bar(np.arange(len(rows)), values, color="#E45756")
    ax.set_xticks(np.arange(len(rows)))
    ax.set_xticklabels(labels, rotation=60, ha="right")
    ax.set_ylabel("Count")
    ax.set_title("Top tail-to-head confusions")
    return save_or_placeholder(fig, out_path)


def plot_calibration_grid_heatmap(calibration_rows, out_path):
    plt = get_matplotlib_pyplot()
    if plt is None:
        return write_figure_placeholder(out_path, "matplotlib unavailable")
    if not calibration_rows:
        return write_figure_placeholder(out_path, "no calibration grid rows available")
    tau_values = sorted({maybe_float(row.get("tau_norm")) for row in calibration_rows})
    la_values = sorted({maybe_float(row.get("logit_adjust_tau")) for row in calibration_rows})
    heatmap = np.full((len(tau_values), len(la_values)), np.nan, dtype=np.float64)
    for row in calibration_rows:
        i = tau_values.index(maybe_float(row.get("tau_norm")))
        j = la_values.index(maybe_float(row.get("logit_adjust_tau")))
        score = maybe_float(row.get("selection_score", row.get("val_selection_score")), np.nan)
        if np.isnan(heatmap[i, j]) or score > heatmap[i, j]:
            heatmap[i, j] = score
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(heatmap, aspect="auto", origin="lower", cmap="viridis")
    ax.set_xticks(np.arange(len(la_values)))
    ax.set_xticklabels([str(v) for v in la_values])
    ax.set_yticks(np.arange(len(tau_values)))
    ax.set_yticklabels([str(v) for v in tau_values])
    ax.set_xlabel("Logit adjustment tau")
    ax.set_ylabel("Tau normalization")
    ax.set_title("Calibration validation score")
    fig.colorbar(im, ax=ax, label="selection score")
    return save_or_placeholder(fig, out_path)


def plot_feature_pca(model, loader, cls_num_list, out_path, device=DEVICE, max_images=2000):
    plt = get_matplotlib_pyplot()
    if plt is None:
        return write_figure_placeholder(out_path, "matplotlib unavailable")
    features = []
    labels = []
    model.eval()
    with torch.no_grad():
        for image, target in loader:
            image = image.to(device, non_blocking=True)
            _, feat = model(image, return_features=True)
            features.append(feat.detach().cpu())
            labels.append(target.detach().cpu())
            if sum(x.size(0) for x in features) >= max_images:
                break
    if not features:
        return write_figure_placeholder(out_path, "no features available")
    x = torch.cat(features, dim=0)[:max_images].numpy()
    y = torch.cat(labels, dim=0)[:max_images].numpy()
    x = x - x.mean(axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(x, full_matrices=False)
    coords = x @ vt[:2].T
    groups = get_frequency_groups(cls_num_list)
    group_per_class = groups["group_per_class"]
    fig, ax = plt.subplots(figsize=(6, 5))
    for group, color in [("head", "#4C78A8"), ("medium", "#F58518"), ("tail", "#54A24B")]:
        mask = np.array([group_per_class[int(label)] == group for label in y])
        if np.any(mask):
            ax.scatter(coords[mask, 0], coords[mask, 1], s=8, alpha=0.55, label=group, color=color)
    ax.set_title("Feature PCA by frequency group")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend(markerscale=2)
    return save_or_placeholder(fig, out_path)


def plot_qualitative_tail_examples(model, loader, class_names, cls_num_list, out_path,
                                   device=DEVICE, calibration=None, max_images=12):
    plt = get_matplotlib_pyplot()
    if plt is None:
        return write_figure_placeholder(out_path, "matplotlib unavailable")
    groups = get_frequency_groups(cls_num_list)
    tail_set = set(map(int, groups["tail"]))
    examples = []
    model.eval()
    with torch.no_grad():
        for image, target in loader:
            image_device = image.to(device, non_blocking=True)
            logits = model(image_device)
            if calibration is not None:
                logits = calibrate_logits_groupwise(logits, cls_num_list, calibration, groups)
            prob = torch.softmax(logits, dim=1)
            conf, pred = prob.max(dim=1)
            for idx in range(image.size(0)):
                true_cls = int(target[idx])
                if true_cls not in tail_set:
                    continue
                examples.append({
                    "image": inverse_normalize(image[idx]).clamp(0, 1),
                    "true": true_cls,
                    "pred": int(pred[idx].detach().cpu()),
                    "conf": float(conf[idx].detach().cpu()),
                })
                if len(examples) >= max_images:
                    break
            if len(examples) >= max_images:
                break
    if not examples:
        return write_figure_placeholder(out_path, "no tail examples available")
    cols = min(4, len(examples))
    rows = int(math.ceil(len(examples) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.asarray(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, ex in zip(axes, examples):
        img = ex["image"].permute(1, 2, 0).numpy()
        ax.imshow(img)
        true_name = class_names[ex["true"]] if ex["true"] < len(class_names) else str(ex["true"])
        pred_name = class_names[ex["pred"]] if ex["pred"] < len(class_names) else str(ex["pred"])
        ax.set_title(f"T:{true_name}\nP:{pred_name} {ex['conf']:.2f}", fontsize=8)
        ax.axis("off")
    return save_or_placeholder(fig, out_path)


def generate_saved_figures_for_scenario(imb_type, imb_factor):
    out_dir = selected_dir(imb_type, imb_factor) / "figures"
    out_dir.mkdir(parents=True, exist_ok=True)
    selected_config_path = selected_dir(imb_type, imb_factor) / "selected_config.json"
    if not selected_config_path.exists():
        write_figure_placeholder(out_dir / "class_distribution.png", "selected_config.json not found")
        return []
    config = read_json(selected_config_path)
    cls_num_list = config.get("train_cls_num_list")
    class_names = config.get("class_names", [str(i) for i in range(len(cls_num_list or []))])
    if not cls_num_list:
        write_figure_placeholder(out_dir / "class_distribution.png", "train class counts unavailable")
        return []
    groups = get_frequency_groups(cls_num_list)
    created = [plot_class_distribution(cls_num_list, groups, out_dir / "class_distribution.png")]

    per_class_path = selected_dir(imb_type, imb_factor) / "per_class_test.csv"
    if not per_class_path.exists():
        per_class_path = selected_dir(imb_type, imb_factor) / "per_class_val.csv"
    per_class_rows = read_csv_rows(per_class_path)
    if per_class_rows:
        acc = [maybe_float(row.get("per_class_acc"), np.nan) for row in sorted(per_class_rows, key=lambda r: maybe_int(r.get("class_id")))]
        created.append(plot_per_class_acc_vs_freq(acc, cls_num_list, groups, out_dir / "per_class_acc_vs_freq.png"))
    else:
        created.append(write_figure_placeholder(out_dir / "per_class_acc_vs_freq.png", "per-class accuracy CSV unavailable"))

    confusion_path = selected_dir(imb_type, imb_factor) / "confusion_test.npy"
    if not confusion_path.exists():
        confusion_path = selected_dir(imb_type, imb_factor) / "confusion_val.npy"
    if confusion_path.exists():
        created.append(plot_tail_to_head_confusion(np.load(confusion_path), groups, class_names, out_dir))
    else:
        created.append(write_figure_placeholder(out_dir / "confusion_tail_to_head.png", "confusion matrix unavailable"))

    calibration_grid_path = Path(config.get("run_dir", "")) / "calibration_grid.csv"
    if not calibration_grid_path.exists():
        calibration_grid_path = RESULTS_DIR / "calibration_summary.csv"
    created.append(plot_calibration_grid_heatmap(read_csv_rows(calibration_grid_path), out_dir / "calibration_grid_heatmap.png"))

    # Qualitative examples and feature PCA need a loaded model and data loader; Step 5 exposes functions but does not run them here.
    created.append(write_figure_placeholder(out_dir / "qualitative_tail_examples.png", "call plot_qualitative_tail_examples with a loaded model and loader"))
    created.append(write_figure_placeholder(out_dir / "feature_pca.png", "call plot_feature_pca with a loaded model and loader"))
    return created


def generate_all_saved_figures():
    created = []
    for imb_type, imb_factor in SCENARIOS:
        created.extend(generate_saved_figures_for_scenario(imb_type, imb_factor))
    return created


### Report Notes and Final Artifact Generation

`generate_report_notes()` writes a report scaffold that references the saved result files. `prepare_report_artifacts()` is the one-call helper to refresh tables, figures, final weight copies, and report notes after experiments finish.


In [ ]:
def result_presence_summary():
    files = {
        "final_test_summary": RESULTS_DIR / "final_test_summary.csv",
        "validation_candidate_summary": RESULTS_DIR / "validation_candidate_summary.csv",
        "validation_pareto_summary": RESULTS_DIR / "validation_pareto_summary.csv",
        "seed_confirm_summary": RESULTS_DIR / "seed_confirm_summary.csv",
        "ablation_summary": RESULTS_DIR / "ablation_summary.csv",
        "per_class_all_scenarios": RESULTS_DIR / "per_class_all_scenarios.csv",
        "calibration_summary": RESULTS_DIR / "calibration_summary.csv",
        "selected_configs": RESULTS_DIR / "selected_configs.json",
    }
    return {name: path.exists() for name, path in files.items()}


def markdown_table_preview(rows, columns, max_rows=8):
    if not rows:
        return "_No rows available yet._"
    rows = rows[:max_rows]
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(col, "")) for col in columns) + " |")
    return "\n".join([header, sep] + body)


def generate_report_notes(path=ROOT_DIR / "report_notes.md"):
    path = Path(path)
    final_rows = read_csv_rows(RESULTS_DIR / "final_test_summary.csv")
    ablation_rows = read_csv_rows(RESULTS_DIR / "ablation_summary.csv")
    seed_rows = read_csv_rows(RESULTS_DIR / "seed_confirm_summary.csv")
    calibration_rows = read_csv_rows(RESULTS_DIR / "calibration_summary.csv")
    presence = result_presence_summary()

    final_preview_cols = ["method", "model", "imb_type", "imb_factor", "overall_acc", "macro_acc", "tail_acc", "FLOPs_M"]
    ablation_preview_cols = ["scenario", "method", "overall_acc", "macro_acc", "tail_acc", "delta_over_CE", "delta_tail_over_CE"]

    text = f"""# Report Notes - EEE6503 Project #3

## Introduction

- Modern visual datasets often have long-tailed class distributions.
- Standard cross-entropy training is biased toward frequent classes.
- This project studies imbalanced CIFAR-100 under four scenarios: `exp/0.1`, `exp/0.01`, `step/0.1`, and `step/0.01`.
- Evaluation considers accuracy, macro/group accuracy, parameters, MACs, and FLOPs.

## Proposed Method

TALC++ is a three-stage long-tail recognition method.

1. Tail-aware representation learning: train a CIFAR backbone with CE strong augmentation, LDAM-DRW, Balanced Softmax, or logit-adjusted CE.
2. BN-safe decoupled classifier adaptation: freeze the learned backbone, keep frozen BatchNorm layers in eval mode, and retrain only the classifier with class-balanced exposure.
3. Validation-gated group-wise calibration: tune group temperatures, post-hoc logit adjustment, and tau-normalization on validation only. The identity setting is always included, so calibration is used only when it improves validation selection score.

## Experimental Setup

- Dataset: CIFAR-100 with the provided `IMBALANCECIFAR100` class.
- Validation split: protected stratified split from the imbalanced training split only.
- Default model: `CifarResNet32`.
- Training budget: `stage1 + stage2 + bn_align <= 200` epochs; batch size fixed at 128.
- Test split: used only by the guarded final frozen evaluation cell after selected configs are fixed.
- Final track: `{FINAL_SUBMISSION_TRACK}`.

## Quantitative Results

Final test summary file: `results/final_test_summary.csv`.

{markdown_table_preview(final_rows, final_preview_cols)}

Ablation summary file: `results/ablation_summary.csv`.

{markdown_table_preview(ablation_rows, ablation_preview_cols)}

Seed confirmation file: `results/seed_confirm_summary.csv`.

- Rows available: {len(seed_rows)}
- Final seed should be selected by validation score only, never test score.

Calibration summary file: `results/calibration_summary.csv`.

- Rows available: {len(calibration_rows)}
- Identity/no-calibration is included in every calibration grid.

## Qualitative Results

Expected figure files under each `logs/scenario_<type>_<factor>/selected/figures/` directory:

- `class_distribution.png`
- `per_class_acc_vs_freq.png`
- `confusion_tail_to_head.png`
- `qualitative_tail_examples.png`
- `feature_pca.png`
- `calibration_grid_heatmap.png`

If a figure cannot be generated in the current environment, the notebook writes a `.txt` placeholder next to the target image path.

## Analysis

- Compare CE/CE strong augmentation against LDAM-DRW and Balanced Softmax to identify the strongest representation learner.
- Compare Stage 1 only against `TALC++_no_calib` to isolate BN-safe classifier adaptation.
- Compare `TALC++_no_calib` against `TALC++` to isolate validation-gated calibration.
- Inspect macro and tail accuracy together with overall accuracy to avoid over-correcting toward tail classes at a large overall cost.
- Use validation coverage when interpreting the hardest `exp/0.01` scenario because tail validation estimates can be noisy.

## Concluding Remarks

TALC++ is designed to improve long-tail robustness while keeping computation controlled. Its main safety mechanisms are validation-only model selection, BN-safe classifier adaptation, and calibration that is applied only when validation improves.

## Artifact Checklist

{json.dumps(presence, indent=2)}

## References

- Cao et al., Learning Imbalanced Datasets with Label-Distribution-Aware Margin Loss, NeurIPS 2019.
- Kang et al., Decoupling Representation and Classifier for Long-Tailed Recognition, ICLR 2020.
- Menon et al., Long-tail Learning via Logit Adjustment, ICLR 2021.
- Ren et al., Balanced Meta-Softmax for Long-Tailed Visual Recognition, NeurIPS 2020.
- Zhong et al., Improving Calibration for Long-Tailed Recognition, CVPR 2021.
- Alshammari et al., Long-Tailed Recognition via Weight Balancing, CVPR 2022.
"""
    path.write_text(text)
    return str(path)


def prepare_report_artifacts(generate_figures=True):
    summary = generate_result_tables()
    figure_outputs = generate_all_saved_figures() if generate_figures else []
    report_path = generate_report_notes()
    summary["figure_outputs"] = figure_outputs
    summary["report_notes"] = report_path
    write_json(RESULTS_DIR / "report_artifact_summary.json", summary)
    return summary


def run_static_smoke_verification():
    checks = {
        "batch_size_128": BATCH_SIZE == 128,
        "epoch_budget_ok": TOTAL_FINAL_EPOCHS <= 200,
        "dataset_cifar100": DATASET == "CIFAR100",
        "run_final_test_default_false": RUN_FINAL_TEST is False,
        "scenarios_count": len(SCENARIOS) == 4 or SMOKE_TEST,
        "model_registry_ready": "resnet32" in MODEL_REGISTRY,
        "stage1_function_ready": callable(train_stage1_representation),
        "stage2_function_ready": callable(train_stage2_classifier),
        "calibration_function_ready": callable(tune_calibration),
        "orchestration_function_ready": callable(run_scenario_model_search),
        "report_notes_function_ready": callable(generate_report_notes),
    }
    write_json(RESULTS_DIR / "static_smoke_verification.json", checks)
    print(json.dumps(checks, indent=2))
    assert all(checks.values()), checks
    return checks


# Lightweight report scaffold generation is safe before training results exist.
generate_report_notes()
print("Report scaffold written to report_notes.md. Run prepare_report_artifacts() after experiments finish.")
